# Model Selection and Comparison — Making the Call Like a Professional

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb14_model_selection_protocol_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Declare a **selection protocol before any score exists** — the full roster of reference models from nb01–nb13 (6 candidates for MedScreen, 7 for HomeValue, dummy floors included), the incumbent each stakeholder runs today, the order the challengers climb in, a margin band (small / base / large), and a deployment bar for the test set.
2. Report every candidate's evidence as its **five fold scores, their mean, and their SD** on identical folds — no per-model interval — and check every candidate against the dummy floor.
3. Climb the **champion ladder** with `compare_paired`: a challenger replaces the current champion only when it is better by a worthwhile amount; then run the margin band and name the closest rival and its switching cutoff.
4. Write the **champion statement** — champion, evidence, checks, and what would change your mind — in the stakeholder's language.
5. Open the **locked test set exactly once per case**, for the frozen champion only, refit on `X_train`; judge it against the deployment bar declared in advance, use INSIDE / ABOVE / BELOW only to describe where it landed relative to the CV mean ± δ, and internalize the **singleness rule**: in any real business analysis, the test set opens once.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one champion-pick exercise per case. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Today is the payoff. Thirteen notebooks of locking discipline (test set sealed, all evaluation via cross-validation on the training set, every new model benchmarked against the nb03 / nb06 linear reference) converge into a single structured ceremony: **declare the protocol, climb the ladder, state the champion, open the locked envelope.**

Imagine you are presenting to **the State Health Department's review board** that has to approve MedScreen's breast-cancer screening pipeline, or to **HomeValue Analytics' deployment council** — the CFO and the people who sign off on the property-value tool. Both rooms want the same three things from you:

1. A **defensible champion selection** — not "I tried a bunch of models and this one looked best", but *"every reference model introduced from nb01 onward was declared as a candidate before any score existed and evaluated on the same five folds with one primary metric; starting from the model you run today, each challenger faced the champion of the moment with a paired comparison against a margin you set in advance, and it took the job only if it was better by a worthwhile amount."*
2. A **champion statement** — four points in the room's own language: what we recommend, the evidence (the outcome, the mean gain, the interval corrected for fold overlap, the margin), the checks, and what would change our mind.
3. A **single test-set evaluation** of that frozen champion — the one and only authorized opening of the locked envelope — judged against the deployment bar the room agreed to before the envelope was opened.

The third bullet is the part this notebook makes visible. Before today every cell that touched the test set was off-limits. **In §6 the test set opens — once, for each of the two business cases — and never again in the course.** The test set is for confirming the champion you already chose, not for choosing it.

> **A question that often comes up here:** *"why is opening the test set such a big deal?"* Because every time you open the test set and let what you see influence what you do next, you have leaked the test set into model selection — which means the test set can no longer give you an unbiased estimate of how the model will perform on new data. The discipline is not statistical pedantry; it is the only mechanism by which the test-set point estimate has any meaning at all. nb14's ceremony exists to drive that point home in the most visible way possible: freeze the champion, open the envelope, write the number down, close the envelope, never reopen.

---

## 1. Setup — Imports, Incumbents, Helpers

The setup cell does the same jobs as nb12 / nb13 plus the tools this notebook is built on: **`compare_paired`** (nb08's comparison, pasted unchanged), **`champion_ladder`** (the climb from the incumbent, one `compare_paired` call per challenger), **`plot_fold_strip`** (a picture of each candidate's five fold scores and their mean — no interval), **`plot_ladder`** (each step's mean gain and corrected interval against $\pm\delta$), and **`describe_test`** (INSIDE / ABOVE / BELOW the champion's CV mean $\pm\delta$ — a description for the ceremony, never the ship decision).

> 💡 **Gemini Prompt:** "Set up the standard nb14 imports: pandas, numpy, matplotlib, scipy.stats, `sklearn.base.clone`, sklearn `datasets` (`load_breast_cancer`, `fetch_california_housing`), `model_selection` (`train_test_split`, `cross_val_score`, `StratifiedKFold`, `KFold`), `tree`, `ensemble`, `linear_model` (LogisticRegression, LinearRegression, Ridge, Lasso), `dummy`, `preprocessing.StandardScaler`, `pipeline.Pipeline`, and `metrics` (roc_auc_score, accuracy_score, f1_score, r2_score, mean_squared_error, mean_absolute_error). Suppress warnings, set `pd.set_option('display.precision', 4)`, `plt.rcParams['figure.figsize']=(10,6)`, `RANDOM_SEED=474`, and the color constants `CLF_COLOR`, `REG_COLOR`, `GREY`, `GREEN`, `RED`. Build `reference_clf` (StandardScaler → `LogisticRegression(C=1.0)`) and `reference_reg` (StandardScaler → `LinearRegression()`) — the incumbents. Paste the course's `compare_paired(scores_a, scores_b, margin)` from nb08 unchanged. Add `champion_ladder(fold_scores, ladder, margin)`, which starts from `ladder[0]` and promotes each later challenger only when `compare_paired(challenger, champion, margin)` returns 'A is better by a worthwhile amount', returning the champion and a log DataFrame; `plot_fold_strip` (each candidate's five fold scores as small dots plus their mean as a large dot, no error bars); `plot_ladder` (each log row's mean gain with a horizontal error bar for its interval against a shaded ±δ band); and `describe_test(test_score, cv_mean, margin)` returning 'INSIDE', 'ABOVE', or 'BELOW' the band cv_mean ± margin."
>
> **After running, verify:**
> - [ ] No import errors, and `RANDOM_SEED` is 474
> - [ ] `reference_clf` and `reference_reg` are `Pipeline` objects with `StandardScaler` + the linear estimator
> - [ ] `compare_paired`, `champion_ladder`, `plot_fold_strip`, `plot_ladder`, and `describe_test` are all callable
> - [ ] No helper computes a confidence interval around a single model's score

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             r2_score, mean_squared_error, mean_absolute_error)
import time, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'
GREEN     = '#2ca02c'
RED       = '#d62728'

# --- nb03 / nb06 linear references: the models each stakeholder runs today (the incumbents) ---
reference_clf = Pipeline([('scaler', StandardScaler()),
                          ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
reference_reg = Pipeline([('scaler', StandardScaler()),
                          ('reg',    LinearRegression())])

# --- The course's one comparison tool (nb08). A = challenger, B = current champion; ONE splitter object. ---
def compare_paired(scores_a, scores_b, margin):
    """Paired comparison on identical folds; higher score = better."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    t = stats.t.ppf(0.975, k - 1)
    se = d.std(ddof=1) * np.sqrt(1/k + 1/(k - 1))   # Nadeau–Bengio correction
    lo, hi = d.mean() - t*se, d.mean() + t*se
    if lo > margin:
        verdict = "A is better by a worthwhile amount"
    elif hi < -margin:
        verdict = "B is better by a worthwhile amount"
    elif -margin < lo and hi < margin:
        verdict = "Practically equivalent: choose the simpler model"
    else:
        verdict = "Unresolved: apply the default you declared"
    return d, d.mean(), (lo, hi), verdict

# --- The champion ladder (development only; the test set is never used here) ---
def champion_ladder(fold_scores, ladder, margin):
    """Simplest-first ladder on CV evidence.
    fold_scores: {name: fold scores from ONE shared cv object}; ladder: incumbent first, then the
    challengers in the declared order. A challenger replaces the current champion only on
    'A is better by a worthwhile amount'. Returns (champion, log as a DataFrame)."""
    champion, log = ladder[0], []
    for challenger in ladder[1:]:
        d, gain, (lo, hi), verdict = compare_paired(fold_scores[challenger], fold_scores[champion], margin)
        promoted = verdict.startswith("A is better")
        log.append({"challenger": challenger, "vs": champion, "mean gain": gain,
                    "low": lo, "high": hi, "verdict": verdict, "promoted": promoted})
        if promoted:
            champion = challenger
    return champion, pd.DataFrame(log)

# --- Picture of one model's evidence: its five fold scores and their mean (no interval) ---
def plot_fold_strip(scores_dict, metric_name, title, ax, color=CLF_COLOR, highlight=None):
    """Dot strip: each candidate's five fold scores (small dots) and their mean (large dot)."""
    names = sorted(scores_dict, key=lambda n: np.mean(scores_dict[n]))
    labels = []
    for i, name in enumerate(names):
        s = np.asarray(scores_dict[name])
        c = GREEN if name == highlight else color
        ax.scatter(s, np.full(len(s), i), s=25, color=c, alpha=0.55, zorder=2)
        ax.scatter([s.mean()], [i], s=120, color=c, edgecolor='black', zorder=3)
        labels.append(f"{name}\nmean {s.mean():.4f} · SD {s.std(ddof=1):.4f}")
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel(f'5-fold CV {metric_name}: five fold scores (small dots) and their mean (large dot)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Picture of a ladder: each step's mean gain and corrected interval against ±δ ---
def plot_ladder(log, margin, metric_name, title, ax, color=CLF_COLOR):
    ax.axvspan(-margin, margin, color=GREY, alpha=0.2, label=f'±δ (δ = {margin}, declared before scoring)')
    ax.axvline(0, color='black', linewidth=0.8)
    for i, r in log.iterrows():
        face = color if r['promoted'] else 'white'
        ax.errorbar(r['mean gain'], i, xerr=[[r['mean gain'] - r['low']], [r['high'] - r['mean gain']]],
                    fmt='o', color=color, ecolor=color, capsize=5, markersize=8, markerfacecolor=face)
        ax.text(r['high'], i, '  ' + r['verdict'].split(':')[0] + (' → PROMOTED' if r['promoted'] else ''),
                va='center', fontsize=8)
    ax.set_yticks(range(len(log)))
    ax.set_yticklabels([f"{r['challenger']}\n  vs {r['vs']}" for _, r in log.iterrows()], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(f'Mean gain in {metric_name} (challenger − current champion), interval corrected for fold overlap')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), fontsize=8)
    ax.grid(True, alpha=0.3, axis='x')

# --- Ceremony description: where the test score lands relative to the champion's CV mean ± δ ---
def describe_test(test_score, cv_mean, margin):
    """INSIDE / ABOVE / BELOW the band CV mean ± δ. A description, never the ship decision."""
    if test_score > cv_mean + margin:
        return 'ABOVE'
    if test_score < cv_mean - margin:
        return 'BELOW'
    return 'INSIDE'

print("✓ Setup, incumbents, compare_paired, champion_ladder, plot helpers, describe_test — all loaded")

## 2. Load Both Datasets — One Split Each, Two Locked Envelopes

Same first split as every notebook since nb01 — `train_test_split(..., test_size=0.20, random_state=RANDOM_SEED)`, stratified for MedScreen — and, since nb09, nothing else: no validation set, no second split. Because the first call is unchanged, the test envelopes hold exactly the same rows they have held all term. Every model choice below comes from five folds of `X_train`; the envelopes open in §6.

> 💡 **Gemini Prompt:** "Apply the course's single 80/20 split to both demo datasets. Load Wisconsin Breast Cancer with `load_breast_cancer(as_frame=True)` and California Housing with `fetch_california_housing(as_frame=True)`. For each, call `train_test_split(X, y, test_size=0.20, random_state=RANDOM_SEED)` once — add `stratify=y` for the classification case — to get `X_train_*` and the LOCKED `X_test_*`. Do not split again. Build `cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)` and `cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)`. Print the train and test sizes per case with a note that the test envelopes stay locked until §6."
>
> **After running, verify:**
> - [ ] Classification: 455 training patients and 114 locked test patients
> - [ ] Regression: 16,512 training tracts and 4,128 locked test tracts
> - [ ] `cv_clf` is a `StratifiedKFold(n_splits=5)` and `cv_reg` is a `KFold(n_splits=5)`
> - [ ] There is no validation set and no second split; both `X_test_*` envelopes stay unused until §6

In [ ]:
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target

# ONE split per dataset, 80/20 — the same first call as nb01-nb13, so the locked test rows are unchanged.
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification: X_train {len(X_train_clf):>6,} | X_test {len(X_test_clf):>5,} (LOCKED until §6.1)')
print(f'Regression:     X_train {len(X_train_reg):>6,} | X_test {len(X_test_reg):>5,} (LOCKED until §6.2)')
print('Five folds for every candidate: cv_clf (stratified) and cv_reg — the same objects for every comparison.')

## 3. The Model Selection Problem

Picking a model is harder than it sounds. The naïve approach — *"just compare the numbers and pick the highest one"* — has six classic failure modes. The right approach has six corresponding fixes:

| What goes wrong | What we do instead |
|---|---|
| Different CV splits per model (apples vs oranges) | **Same five folds** for every candidate — one splitter object (apples vs apples) |
| Different metric per model ("metric shopping") | **One declared primary metric**; supporting metrics are reported as checks |
| Pick the model by test-set score | Pick by cross-validation; **the test set confirms the frozen champion against a bar declared in advance** |
| Try new candidates until something wins | **Lock the roster, the ladder order and the margins before fitting any model** |
| Crown the largest mean, and let fit time break near-ties | **Climb a ladder from the model the stakeholder runs today**: a challenger takes the job only when `compare_paired` shows it is better by a worthwhile amount. Fit time and supporting metrics are reported as checks; they never vote |
| No written rationale | A **four-point champion statement** in the stakeholder's language |

The protocol below operationalizes the right column. **Section 4** declares the criteria block — roster, ladder order, margin band, deployment bar — before any results are seen; **Section 5** scores every candidate on identical folds (5.1) and climbs the ladder (5.2); the **PAUSE-AND-DO exercises** check each champion the way its stakeholder will and write the champion statement; **Section 6** opens the locked test set for the frozen champions.

> **A question that often comes up here:** *"isn't this overkill for a small problem? Can't I just compare scores and pick the highest?"* In a class assignment, sure — picking the highest score is fast, and the cost of being wrong is a grade. In a real business decision, *"I picked the highest score"* is the answer that loses you the room. The review board will ask which models you compared, on what data, with what metric, and how you decided the difference was real and large enough to matter. A stakeholder who has lived through a model that looked great in development and embarrassed the company in production will not accept a one-sentence justification. The six-row table above is what a defensible answer looks like — and writing the protocol down *before* you fit any model is what keeps you honest later, when the numbers tempt you to skip steps.

---

## 4. Declare the Criteria Block — Roster, Ladder, Margins, Bar

Everything that decides the outcome is written down here, **before any score exists**. This rules out "model shopping" — the failure mode where you keep trying new models, metrics, or margins until one happens to favor your preferred answer. **Every reference model introduced from nb01 onward is on the roster, and each one carries the configuration its source notebook carried forward**. nb14 is the chain's audit, not a re-tuning step.

| # | Family | Source notebook | MedScreen (classification) | HomeValue (regression) |
|---|---|---|---|---|
| 1 | **Dummy floor** | nb03 (reg), nb06 (clf) | `DummyClassifier('most_frequent')` | `DummyRegressor('mean')` |
| 2 | **Linear incumbent** — the model each stakeholder runs today | nb06 (clf), nb03 (reg) | `LogReg(C=1.0)` | OLS (`LinearRegression`) |
| 3 | **Ridge regularization** (L2) | nb05 / nb08 | — *(L2 is already the default for LogReg)* | `Ridge(alpha=1.0)` |
| 4 | **Sparse linear** (L1) | nb05 (reg), nb09 (clf) | `LogReg L1(C=0.1)` | `Lasso(alpha=0.01)` |
| 5 | **Single tree** | nb11 | `Tree(depth=3)` | `Tree(depth=5)` |
| 6 | **Bagged ensemble** | nb12 | `Forest()` — scikit-learn's defaults (100 trees) | `Forest()` — scikit-learn's defaults (100 trees) |
| 7 | **Boosted ensemble** | nb13 | `GBM()` — scikit-learn's defaults | `GBM(learning_rate=0.2, n_estimators=200)` |

Three notes on the roster. The **dummy floor** is not a candidate for the job; it is the line every candidate must clear on every fold before anything else is discussed (ROC-AUC 0.500 is random ranking; R² ≈ 0 is predicting the mean). The **forests are scikit-learn's defaults** because that is what nb12 carried forward: in both cases the grid's best cell was a nominee that did not show a worthwhile gain over the default forest. The **boosted models** follow nb13 the same way: for MedScreen the grid's nominee did not earn a worthwhile gain over the default GBM, so the default is carried; for HomeValue it did, so the tuned `(learning_rate=0.2, n_estimators=200)` model is carried — and because a grid search picked that configuration on folds like these, its fold scores here are a little **optimistic**. That is a reason for caution when you read its mean, not a reason to drop it from the roster.

### The criteria block, one per case

| | **MedScreen** — State Health Department review board | **HomeValue** — the CFO and the deployment council |
|---|---|---|
| Decision | Which model ranks biopsies for further review (the operating threshold stays nb07's separate, cost-based decision) | Which model produces the neighborhood value estimates (the decision uses the point estimate) |
| Primary metric | ROC-AUC (accuracy and F1 reported as checks) | $R^2$ (RMSE and MAE in dollars reported as checks) |
| Incumbent — the starting champion | `LogReg(C=1.0)` from nb06 | OLS from nb03 |
| Ladder order — challengers by operational burden | L1 → tree → forest → GBM | Ridge → Lasso → tree → forest → GBM |
| Margin band — **base decides** | 0.005 / **0.01** / 0.02 ROC-AUC (one more correctly ordered malignant–benign pair per 200 / **100** / 50) | 0.005 / **0.01** / 0.02 $R^2$ (nb08 translated these into dollars of typical error; Exercise 2 does it again at the champion's level) |
| Role of predictive uncertainty | Guardrails that can block a promotion, never create one: malignant recall at the nb07 operating threshold must stay non-inferior to the **fixed** incumbent, and the Brier score of the probabilities must not get worse — each within a tolerance the board fixes before the comparison. Reported as checks in Exercise 1 | Not used here: the CFO acts on the point estimate, not on a price range, so a prediction-range guardrail does not apply to this decision |
| Deployment bar for the one test-set opening | test ROC-AUC **≥ 0.95** | test $R^2$ **≥ 0.75** |

These are the same margins nb08 declared and nb09–nb13 used, unchanged. Writing them down again here, before the scores, is what makes them a policy rather than a rationalization.

> **A question that often comes up here:** *"do I have to memorize where every hyperparameter came from?"* No — the roster is the **audit trail**, not a quiz. Its job is to let anyone in the room trace each candidate back to the notebook that produced it, so the comparison is reproducible from scratch. When the deployment council asks *"why is the GBM's `learning_rate` 0.2 and not 0.1?"* you point to nb13, where that setting earned a worthwhile gain over the default; when the review board asks *"why the default forest and not a tuned one?"* you point to nb12, where the tuned nominee did not. What matters in the meeting is the criteria block — the incumbent, the order, the margins, and the bar — because that is what decides the outcome.

> 💡 **Gemini Prompt:** "Declare two ordered dictionaries — `clf_models` and `reg_models` — that lock the candidate roster before any results are seen. **Classification (6):** `DummyClassifier('most_frequent')`, the `reference_clf` incumbent, a StandardScaler + `LogisticRegression(penalty='l1', C=0.1, solver='liblinear')` pipeline, `DecisionTreeClassifier(max_depth=3)`, `RandomForestClassifier(n_jobs=-1)`, and `GradientBoostingClassifier()`. **Regression (7):** `DummyRegressor('mean')`, the `reference_reg` incumbent, StandardScaler pipelines with `Ridge(alpha=1.0)` and `Lasso(alpha=0.01)`, `DecisionTreeRegressor(max_depth=5)`, `RandomForestRegressor(n_jobs=-1)`, and `GradientBoostingRegressor(learning_rate=0.2, n_estimators=200)`. Use `random_state=RANDOM_SEED` where applicable. Then declare the criteria block: `LADDER_CLF` and `LADDER_REG` as the roster names in order without the dummy (incumbent first); `DELTAS_AUC` and `DELTAS_R2` as `{'small': 0.005, 'base': 0.01, 'large': 0.02}` with `DELTA_AUC` / `DELTA_R2` the base values; `DEPLOY_BAR_CLF = 0.95` and `DEPLOY_BAR_REG = 0.75`. Print the roster sizes, both ladders, the margins, and the bars."
>
> **After running, verify:**
> - [ ] `clf_models` has exactly 6 entries and `reg_models` exactly 7
> - [ ] Each ladder starts with the incumbent (`LogReg C=1`, OLS) and excludes the dummy
> - [ ] The margins are 0.005 / 0.01 / 0.02 in both cases, with the base margin 0.01
> - [ ] The deployment bars are 0.95 ROC-AUC and 0.75 R²
> - [ ] No candidate, margin, or bar is changed below this cell

In [ ]:
# --- Candidate roster, locked before any score is computed ---
clf_models = {
    'Dummy (most_frequent)':          DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED),
    'LogReg C=1 [nb06 incumbent]':    reference_clf,
    'LogReg L1 (C=0.1)':              Pipeline([('scaler', StandardScaler()),
                                                ('clf', LogisticRegression(penalty='l1', C=0.1, solver='liblinear',
                                                                           random_state=RANDOM_SEED, max_iter=5000))]),
    'Tree (depth=3) [nb11]':          DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'Forest (sklearn default) [nb12]': RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1),
    'GBM (sklearn default) [nb13]':   GradientBoostingClassifier(random_state=RANDOM_SEED),
}
reg_models = {
    'Dummy (mean)':                   DummyRegressor(strategy='mean'),
    'OLS [nb03 incumbent]':           reference_reg,
    'Ridge (alpha=1)':                Pipeline([('scaler', StandardScaler()),
                                                ('reg', Ridge(alpha=1.0, random_state=RANDOM_SEED))]),
    'Lasso (alpha=0.01)':             Pipeline([('scaler', StandardScaler()),
                                                ('reg', Lasso(alpha=0.01, random_state=RANDOM_SEED, max_iter=5000))]),
    'Tree (depth=5) [nb11]':          DecisionTreeRegressor(max_depth=5, random_state=RANDOM_SEED),
    'Forest (sklearn default) [nb12]': RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1),
    'GBM (lr=0.2, n=200) [nb13]':     GradientBoostingRegressor(learning_rate=0.2, n_estimators=200,
                                                                random_state=RANDOM_SEED),
}

# --- The criteria block, declared before scoring ---
# Ladder: the incumbent first, then the challengers in order of operational burden (= roster order).
LADDER_CLF = [n for n in clf_models if not n.startswith('Dummy')]
LADDER_REG = [n for n in reg_models if not n.startswith('Dummy')]
# Margin bands: base decides; small and large are the sensitivity check (the same numbers since nb08).
DELTAS_AUC = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # ROC-AUC: correctly ordered pairs per 100
DELTAS_R2  = {'small': 0.005, 'base': 0.01, 'large': 0.02}   # R²
DELTA_AUC, DELTA_R2 = DELTAS_AUC['base'], DELTAS_R2['base']
# Deployment bars for the one test-set opening per case (§6).
DEPLOY_BAR_CLF = 0.95   # MedScreen: test ROC-AUC at or above 0.95
DEPLOY_BAR_REG = 0.75   # HomeValue: test R² at or above 0.75

print(f"✓ Roster locked: {len(clf_models)} classification and {len(reg_models)} regression candidates (dummy floors included)")
print("MedScreen ladder:", " → ".join(LADDER_CLF))
print("HomeValue ladder:", " → ".join(LADDER_REG))
print(f"Margins — ROC-AUC {DELTAS_AUC} | R² {DELTAS_R2} (base decides)")
print(f"Deployment bars — MedScreen test ROC-AUC ≥ {DEPLOY_BAR_CLF}; HomeValue test R² ≥ {DEPLOY_BAR_REG}")

## 5. The Evidence, Then the Ladder

### 5.1 Every candidate on the same five folds

For each case, evaluate every candidate on the **same** five folds of `X_train` — the `cv_clf` and `cv_reg` objects from §2. Track the primary metric fold by fold, plus supporting metrics that become checks in the champion statement:

- **MedScreen:** ROC-AUC (primary, nb07), accuracy (nb06), F1 (nb07).
- **HomeValue:** $R^2$ (primary, nb03), RMSE and MAE (nb03), reported in dollars.

Every metric is one the course has already presented. The code walks the folds manually with `cv_clf.split(X, y)` / `cv_reg.split(X, y)` and calls **nb03's `mean_absolute_error`, `mean_squared_error`, and `r2_score`** (plus **nb06's `accuracy_score`** and **nb07's `roc_auc_score` and `f1_score`**) directly on `(y_true, y_pred)` for each fold, keeping every fold array — the ladder in §5.2 needs them.

What gets reported for each candidate is what nb08 taught: **its five fold scores, their mean, and their SD**, and a strip of the fold scores as the picture. There is no interval around any single model's score, and nothing in this section ranks or picks anything. Before moving on, check one thing: every candidate on the ladder must beat the dummy floor on every fold.

> **A question that often comes up here:** *"why three metrics per case instead of just the primary one?"* Because the primary metric drives the **decision**, but the supporting metrics catch **blind spots** the primary one cannot see. On classification, ROC-AUC scores how well the model ranks malignant biopsies above benign ones — yet two models with similar AUC can behave very differently at an operating threshold; accuracy and F1 expose that. On regression, $R^2$ scores how much variance the model explains — yet two models with similar $R^2$ can have different typical errors in dollars; RMSE and MAE expose that. The supporting metrics never vote; they go on the **Checks** line of the champion statement, where a stakeholder can see at a glance that the choice hides no trade-off.

> 💡 **Gemini Prompt:** "Run a 5-fold CV evaluation of every candidate in `clf_models` and `reg_models` without sklearn's `neg_*` scoring strings. Walk the folds manually with `cv_clf.split(X_train_clf, y_train_clf)` and `cv_reg.split(X_train_reg, y_train_reg)`; for each (model, fold), fit a `clone` of the model on the fold's training rows and predict its held-out rows. **Classification:** `roc_auc_score` on `predict_proba(...)[:, 1]`, plus `accuracy_score` and `f1_score` on `predict(...)`. **Regression:** `r2_score`, `mean_absolute_error`, and `np.sqrt(mean_squared_error(...))`. Keep each metric's five-value fold array and the mean fit time. Store `clf_results` / `reg_results` DataFrames indexed by model name with the primary metric's fold array, mean and SD (`ddof=1`), the supporting metrics' means, and the fit time — in roster order, not sorted. Print a display table per case with the five fold scores, mean, SD, supporting metrics (regression errors multiplied by 100,000 into dollars), and fit time. Check and print whether every non-dummy candidate beats the dummy on every fold. Finally, draw `plot_fold_strip` for the ladder candidates of each case side by side."
>
> **After running, verify:**
> - [ ] Two tables print, with five fold scores, mean, and SD per candidate — no interval column anywhere
> - [ ] RMSE and MAE are in dollars, as positive numbers
> - [ ] The dummy-floor check prints for both cases
> - [ ] Both strips render: five small dots and one large mean dot per candidate, no error bars

In [ ]:
# §5.1 — Every candidate on the SAME five folds. We walk cv_clf / cv_reg manually and call the
# nb03/nb06/nb07 metric functions directly on (y_true, y_pred), keeping every fold array.

def _evaluate_clf_per_fold(model, X, y, cv):
    """Per-fold ROC-AUC (nb07), accuracy (nb06), F1 (nb07) + mean fit time."""
    auc_f, acc_f, f1_f, fit_t = [], [], [], []
    for tr_idx, va_idx in cv.split(X, y):
        m = clone(model)
        t0 = time.time()
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        fit_t.append(time.time() - t0)
        y_va = y.iloc[va_idx]
        auc_f.append(roc_auc_score(y_va, m.predict_proba(X.iloc[va_idx])[:, 1]))
        y_pred = m.predict(X.iloc[va_idx])
        acc_f.append(accuracy_score(y_va, y_pred))
        f1_f.append(f1_score(y_va, y_pred))
    return np.array(auc_f), np.array(acc_f), np.array(f1_f), float(np.mean(fit_t))

def _evaluate_reg_per_fold(model, X, y, cv):
    """Per-fold R² (nb03), RMSE = sqrt(MSE) (nb03), MAE (nb03) + mean fit time."""
    r2_f, rmse_f, mae_f, fit_t = [], [], [], []
    for tr_idx, va_idx in cv.split(X, y):
        m = clone(model)
        t0 = time.time()
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        fit_t.append(time.time() - t0)
        y_va, y_pred = y.iloc[va_idx], m.predict(X.iloc[va_idx])
        r2_f.append(r2_score(y_va, y_pred))
        rmse_f.append(np.sqrt(mean_squared_error(y_va, y_pred)))
        mae_f.append(mean_absolute_error(y_va, y_pred))
    return np.array(r2_f), np.array(rmse_f), np.array(mae_f), float(np.mean(fit_t))

clf_rows, reg_rows = [], []
for name, model in clf_models.items():
    auc, acc, f1, ft = _evaluate_clf_per_fold(model, X_train_clf, y_train_clf, cv_clf)
    clf_rows.append({'Model': name, 'roc_auc_folds': auc, 'roc_auc_mean': auc.mean(), 'roc_auc_sd': auc.std(ddof=1),
                     'accuracy_mean': acc.mean(), 'f1_mean': f1.mean(), 'fit_time_s': ft})
for name, model in reg_models.items():
    r2, rmse, mae, ft = _evaluate_reg_per_fold(model, X_train_reg, y_train_reg, cv_reg)
    reg_rows.append({'Model': name, 'r2_folds': r2, 'r2_mean': r2.mean(), 'r2_sd': r2.std(ddof=1),
                     'rmse_mean': rmse.mean(), 'mae_mean': mae.mean(), 'fit_time_s': ft})
clf_results = pd.DataFrame(clf_rows).set_index('Model')
reg_results = pd.DataFrame(reg_rows).set_index('Model')

# Display: fold scores, mean and SD for the primary metric; supporting metrics as means; fit time
clf_display = pd.DataFrame({
    'ROC-AUC folds': clf_results['roc_auc_folds'].apply(lambda a: ' '.join(f'{x:.4f}' for x in a)),
    'mean':          clf_results['roc_auc_mean'].map('{:.4f}'.format),
    'SD':            clf_results['roc_auc_sd'].map('{:.4f}'.format),
    'accuracy':      clf_results['accuracy_mean'].map('{:.4f}'.format),
    'F1':            clf_results['f1_mean'].map('{:.4f}'.format),
    'fit (s)':       clf_results['fit_time_s'].map('{:.3f}'.format),
})
reg_display = pd.DataFrame({
    'R² folds':      reg_results['r2_folds'].apply(lambda a: ' '.join(f'{x:.4f}' for x in a)),
    'mean':          reg_results['r2_mean'].map('{:.4f}'.format),
    'SD':            reg_results['r2_sd'].map('{:.4f}'.format),
    'RMSE (USD)':    (reg_results['rmse_mean'] * 100_000).map('{:,.0f}'.format),
    'MAE (USD)':     (reg_results['mae_mean'] * 100_000).map('{:,.0f}'.format),
    'fit (s)':       reg_results['fit_time_s'].map('{:.3f}'.format),
})
with pd.option_context('display.max_colwidth', 60, 'display.width', 200):
    print("=== MedScreen (classification) — 5-fold CV on X_train, roster order ===")
    print(clf_display.to_string())
    print("\n=== HomeValue (regression) — 5-fold CV on X_train, roster order ===")
    print(reg_display.to_string())

# The dummy floor: every candidate on the ladder must clear it on every fold
for label, res, col, floor_name in [('MedScreen', clf_results, 'roc_auc_folds', 'Dummy (most_frequent)'),
                                    ('HomeValue', reg_results, 'r2_folds', 'Dummy (mean)')]:
    floor = res.loc[floor_name, col]
    ok = all((res.loc[n, col] > floor).all() for n in res.index if n != floor_name)
    print(f"\n{label}: dummy floor mean {floor.mean():.4f} — every ladder candidate above it on every fold: {ok}")

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
plot_fold_strip({n: clf_results.loc[n, 'roc_auc_folds'] for n in LADDER_CLF}, 'ROC-AUC',
                'MedScreen — fold scores of every ladder candidate', axes[0], color=CLF_COLOR)
plot_fold_strip({n: reg_results.loc[n, 'r2_folds'] for n in LADDER_REG}, 'R²',
                'HomeValue — fold scores of every ladder candidate', axes[1], color=REG_COLOR)
fig.suptitle('The evidence, described: five fold scores per candidate (the dummy floors are in the tables)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Reading the output:**

**The dummy floor anchors both cases.** `DummyClassifier('most_frequent')` scores ROC-AUC **0.500** on every fold — random ranking — and `DummyRegressor('mean')` lands at $R^2$ ≈ **0.000**. The check line prints **True** for both cases: every candidate on either ladder beats its floor on every one of the five folds, so the work is buying real signal.

**MedScreen.** On the 455 training patients the incumbent `LogReg C=1` scores **0.9923, 0.9969, 0.9995, 0.9995, 0.9825** (mean **0.9941**, SD **0.0072**) — the highest mean on the roster. The others trail it: L1 at 0.9920 (SD 0.0124), the forest at 0.9883, the GBM at 0.9872, and the depth-3 tree well behind at 0.9399 with by far the widest spread (SD 0.0514). Notice how high and how tight the top of the strip is: two of the incumbent's folds already sit at 0.9995.

**HomeValue.** The three linear models cluster near $R^2$ 0.60 — OLS and Ridge are identical to four decimals at **0.6044**, Lasso at 0.6022 — and the depth-5 tree reaches 0.6109. Then comes a jump: the forest at **0.8036** (SD 0.0117; typical error, RMSE, about \$51,000 against OLS's \$72,400) and the tuned GBM at **0.8217** (SD 0.0094), the highest mean on the roster.

If you ranked by mean, you would ship the tuned GBM for HomeValue. The strip cannot tell you whether that is right, because it answers *"how did each model score?"* and the stakeholder's question is *"is this model better than the one we run today by enough to matter?"* That is a question about paired gains, and §5.2 asks it one rung at a time.

> **A question that often comes up here:** *"why not just pick the model with the highest mean?"* Because the largest of seven means is optimistic almost by construction: when several candidates are close, the one that comes out on top got a little lucky on these particular folds, and the more candidates you compare, the more that luck inflates the winner — the "optimizer's curse" (Smith and Winkler, 2006). The tuned GBM carries a second dose of the same optimism, because nb13's grid search picked its configuration for scoring well on folds like these (Cawley and Talbot, 2010). And a model switch is a decision with costs — re-validation, retraining, explaining a new model — not a contest. So the course starts from the model the stakeholder already runs and asks each challenger to *earn* the job with a gain the stakeholder said would matter.

---

### 5.2 Climb the ladder — and freeze the champion

The criteria block already says how the champion is chosen; this cell only carries it out:

1. **Start from the incumbent** — `LogReg C=1` for MedScreen, OLS for HomeValue. It is the champion until something beats it.
2. **Take the challengers in the declared order.** Each one faces the *current* champion with `compare_paired(challenger_folds, champion_folds, base δ)`, on the same five folds.
3. **Promote only on "A is better by a worthwhile amount".** Practically equivalent, unresolved, or worse: the champion stays.
4. **Keep the log** — every comparison, its mean gain, its interval corrected for fold overlap, and its outcome. The log is the evidence behind the champion statement.
5. **Freeze the champion.** The model that comes out of the base-margin ladder is the one §6 opens the test set for. Nothing after this cell selects a model.

The figure draws one row per comparison: the mean gain as a dot, the corrected interval as a bar, the gray band as $\pm\delta$. A filled dot marks a promotion.

> 💡 **Gemini Prompt:** "Build `clf_scores` and `reg_scores`, dictionaries from each ladder name to its primary-metric fold array in `clf_results` / `reg_results`. Run `champion_ladder(clf_scores, LADDER_CLF, DELTA_AUC)` and `champion_ladder(reg_scores, LADDER_REG, DELTA_R2)`, storing the champions as `champ_clf_name` / `champ_reg_name` and the logs as `ladder_log_clf` / `ladder_log_reg`. Print both logs with 4-decimal gains and limits. Freeze the champions: store each champion's fold array (`champ_clf_folds`, `champ_reg_folds`) and mean (`champ_clf_mean`, `champ_reg_mean`) and print a FROZEN line per case. Draw `plot_ladder` for both cases side by side."
>
> **After running, verify:**
> - [ ] Each log has one row per challenger, with the champion of the moment in the `vs` column
> - [ ] A row is marked `promoted` only when its outcome is 'A is better by a worthwhile amount'
> - [ ] Two FROZEN lines print, each with the champion's CV mean and SD
> - [ ] Both ladder figures render with the ±δ band

In [ ]:
# §5.2 — Climb each ladder at the BASE margin. The champion that comes out is frozen for §6.
clf_scores = {n: clf_results.loc[n, 'roc_auc_folds'] for n in LADDER_CLF}
reg_scores = {n: reg_results.loc[n, 'r2_folds'] for n in LADDER_REG}

champ_clf_name, ladder_log_clf = champion_ladder(clf_scores, LADDER_CLF, DELTA_AUC)
champ_reg_name, ladder_log_reg = champion_ladder(reg_scores, LADDER_REG, DELTA_R2)

fmt = {c: '{:+.4f}'.format for c in ['mean gain', 'low', 'high']}
with pd.option_context('display.max_colwidth', 60, 'display.width', 220):
    print(f"=== MedScreen ladder (ROC-AUC, δ = {DELTA_AUC}) ===")
    print(ladder_log_clf.to_string(index=False, formatters=fmt))
    print(f"\n=== HomeValue ladder (R², δ = {DELTA_R2}) ===")
    print(ladder_log_reg.to_string(index=False, formatters=fmt))

# FREEZE the champions. §6 consumes these values and selects nothing.
champ_clf_folds = clf_scores[champ_clf_name]
champ_reg_folds = reg_scores[champ_reg_name]
champ_clf_mean, champ_reg_mean = champ_clf_folds.mean(), champ_reg_folds.mean()
print(f"\nFROZEN — MedScreen champion: {champ_clf_name}  (CV ROC-AUC mean {champ_clf_mean:.4f}, SD {champ_clf_folds.std(ddof=1):.4f})")
print(f"FROZEN — HomeValue champion: {champ_reg_name}  (CV R² mean {champ_reg_mean:.4f}, SD {champ_reg_folds.std(ddof=1):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(20, 6.5))
plot_ladder(ladder_log_clf, DELTA_AUC, 'ROC-AUC', 'MedScreen ladder — each challenger vs the champion of the moment',
            axes[0], color=CLF_COLOR)
plot_ladder(ladder_log_reg, DELTA_R2, 'R²', 'HomeValue ladder — each challenger vs the champion of the moment',
            axes[1], color=REG_COLOR)
plt.tight_layout()
plt.show()

**Reading the output:**

**MedScreen — the incumbent keeps the job.** Every challenger's comparison with `LogReg C=1` comes out **unresolved**: L1 (mean gain −0.0022, interval [−0.0129, +0.0086]), the depth-3 tree (−0.0542, [−0.1434, +0.0350]), the forest (−0.0058, [−0.0190, +0.0074]), and the GBM (−0.0070, [−0.0208, +0.0069]). No interval sits above $+0.01$, so nobody is promoted and **`LogReg C=1` is frozen as MedScreen's champion**. In the review board's words: *the evidence does not show a gain large enough to justify any of the more complex models, so under our policy we keep the explainable one.* Notice the L1 row. Earlier versions of this notebook settled the incumbent-versus-L1 pair by fit time — and both fit in a few thousandths of a second, so that was a coin flip. Under the ladder, fit time never votes: the pair is unresolved and the incumbent stays.

**HomeValue — one promotion, then two challengers that do not displace it.** Ridge is **practically equivalent** to OLS (an interval of essentially zero width at zero), Lasso and the depth-5 tree are **unresolved**, and then the forest faces OLS: mean gain **+0.1992** $R^2$, interval **[+0.1800, +0.2183]** — entirely above $+0.01$, **A is better by a worthwhile amount**, and the forest is promoted. From there the forest is the champion of the moment. The tuned GBM — the highest mean on the roster — gains **+0.0181** over it with an interval of **[+0.0091, +0.0271]**: the lower limit falls just short of $+0.01$, so the outcome is **unresolved**. **The default random forest is frozen as HomeValue's champion** — at the CFO's base margin. Exercise 2 shows how close that call is.

> **A question that often comes up here:** *"the tuned GBM has the higher mean — why doesn't an unresolved gain justify switching?"* Because "unresolved" means the evidence cannot tell whether the gain is worth having: an interval running from +0.0091 to +0.0271 is consistent with a gain the CFO said she would act on (above 0.01) and with one she said she would not (just below it). The policy declared before scoring puts the burden of proof on the challenger, and the challenger has not met it — the honest sentence is *"the evidence does not show a gain large enough to justify the tuned GBM, so under our policy we keep the forest,"* never "the two are equivalent". Two more reasons to be careful with this particular challenger: its configuration was picked by a grid search on folds like these, so its edge is optimistic, and an honest correction for that search can only shrink an optimistic estimate (going further: *nested* cross-validation re-runs the whole search inside each outer fold to measure the procedure, not the lucky configuration); and it takes several times longer to fit than the forest, as the tables above show. This one is close, and the honest report says so: Exercise 2 will show that a CFO who switches models more cheaply would have picked the GBM. Because the base margin was declared before scoring, the forest stays — but the CFO should hear that her margin is doing the work here, and she may choose to switch, explicitly and on the record.

---

## 📝 PAUSE-AND-DO Exercise 1 (MedScreen, 8 minutes) — Check the Champion and Write the Statement

§5.2 froze `LogReg C=1` as MedScreen's champion. Before the review board reads a single test-set number, it will ask four questions. Answer them in code, then write the champion statement in the markdown cell below your code.

**Steps:**

1. **The band.** Re-run `champion_ladder(clf_scores, LADDER_CLF, delta)` at each margin in `DELTAS_AUC`. Same champion at all three → *robust across the three declared margins (for this order and these folds)*; otherwise *margin-sensitive* — name the margin that flips it.
2. **The closest rival.** Among the challengers that were not promoted in `ladder_log_clf`, find the one with the highest lower limit (`low`). Print its interval $[L, U]$ and its **switching cutoff**: it would be a worthwhile gain only for a margin below $L$, and practically equivalent only for a margin above $\max(|L|, |U|)$.
3. **The ceiling.** Print the champion's headroom on each fold ($1 - $ fold score) and the interval a **perfect** challenger would get: `compare_paired(np.ones(5), champ_clf_folds, DELTA_AUC)`.
4. **The checks.** Compute the cross-validated **Brier score** — the average squared gap between the predicted probability and what actually happened, lower is better — for the champion and the closest rival with `cross_val_score(..., cv=cv_clf, scoring='neg_brier_score')` (negate it), and print their accuracy, F1, and fit time from `clf_results`.
5. **Write the statement** in the cell below the code.

> 💡 **Gemini Prompt:** "Using `clf_scores`, `LADDER_CLF`, `DELTAS_AUC`, and `champion_ladder`, print the champion at each margin in `DELTAS_AUC` and one line saying whether it is the same at all three. From `ladder_log_clf`, keep the rows with `promoted == False`, pick the row with the largest `low`, and print its challenger, mean gain, interval and outcome, plus the margin below which it would be a worthwhile gain (its `low`, or 'no positive margin' if `low` ≤ 0) and the margin above which it would be practically equivalent (`max(abs(low), abs(high))`). Print `1 - champ_clf_folds` and the result of `compare_paired(np.ones(5), champ_clf_folds, DELTA_AUC)`. For the champion and that rival, compute `-cross_val_score(clf_models[name], X_train_clf, y_train_clf, cv=cv_clf, scoring='neg_brier_score')` and print its mean and folds, plus accuracy, F1, and fit time from `clf_results`."
>
> **After running, verify:**
> - [ ] One line per margin (small, base, large) with the champion it implies, and a robust / margin-sensitive line
> - [ ] The closest rival's interval and both halves of its switching cutoff are printed
> - [ ] The headroom per fold and the perfect challenger's interval and outcome are printed
> - [ ] Brier scores print for two models, with five fold values each

In [ ]:
# YOUR SOLUTION CODE HERE
# MedScreen: check the champion from §5.2 the way the review board will.
#   1. Run the band: champion_ladder(clf_scores, LADDER_CLF, delta) for each delta in DELTAS_AUC.
#      Same champion at all three margins = robust; otherwise margin-sensitive.
#   2. Closest rival: among the challengers that were NOT promoted in ladder_log_clf, the one with
#      the highest lower limit ('low'). Print its interval [L, U] and its switching cutoff:
#      a worthwhile gain only for a margin below L; practically equivalent only for a margin above max(|L|, |U|).
#   3. The ceiling: print the champion's headroom per fold (1 - fold score) and the interval a PERFECT
#      challenger (1.0 on every fold) would get: compare_paired(np.ones(5), champ_clf_folds, DELTA_AUC).
#   4. Checks: cross-validated Brier score for the champion and the closest rival
#      (cross_val_score(..., cv=cv_clf, scoring='neg_brier_score'), negated), plus their accuracy, F1, fit time.
#
# Variables available: clf_scores, LADDER_CLF, DELTAS_AUC, DELTA_AUC, ladder_log_clf, champ_clf_name,
#   champ_clf_folds, clf_models, clf_results, X_train_clf, y_train_clf, cv_clf, compare_paired, champion_ladder

### YOUR CHAMPION STATEMENT — MedScreen

Replace each bracket. When the champion is the incumbent, the **Evidence** line reports its closest rival's comparison; for an unresolved result, use the sentence *"The evidence does not show a gain large enough to justify [challenger], so under our policy we keep [champion]."*

> **Champion.** We recommend **[model]** for **[stakeholder]'s [decision]**.
> **Evidence.** [outcome against the closest rival, on the same five folds: mean gain, interval corrected for fold overlap, the δ we set before comparing — in the board's terms]
> **Checks.** [the small and large margins; the ceiling; Brier score, accuracy, F1, fit time]
> **What would change our mind.** **[runner-up]** is the candidate for a future evaluation (not an automatic fallback); we revisit if **[trigger]**.

## 📝 PAUSE-AND-DO Exercise 2 (HomeValue, 8 minutes) — Check the Champion and Write the Statement

Same template, the regression case. §5.2 froze the forest as HomeValue's champion after it displaced OLS. The deployment council will want the same four answers, in dollars where possible.

**Steps:**

1. **The band.** Re-run `champion_ladder(reg_scores, LADDER_REG, delta)` at each margin in `DELTAS_R2`: robust or margin-sensitive?
2. **The closest rival** among the unpromoted challengers (highest lower limit in `ladder_log_reg`), its interval $[L, U]$, and its switching cutoff.
3. **Dollars.** Translate the champion's promotion into typical error using the CV RMSE of OLS and of the champion (`reg_results['rmse_mean']` is in units of \$100,000). Then translate the three margins and the rival's lower limit $L$ into dollars of typical error **at the champion's level** with `1e5 * (sqrt(v*(1 - R2)) - sqrt(v*(1 - R2 - gain)))`, where `v = y_train_reg.var()` and `R2` is the champion's CV mean. A translation of the metric, not money saved.
4. **The checks.** CV $R^2$ with its SD, RMSE and MAE in dollars, and fit time for the champion and the closest rival.
5. **Write the statement** in the cell below the code. Tie the revisit trigger to housing-market dynamics (income shifts, new construction, a new data release).

> 💡 **Gemini Prompt:** "Using `reg_scores`, `LADDER_REG`, `DELTAS_R2`, and `champion_ladder`, print the champion at each margin and whether it is the same at all three. From `ladder_log_reg`, pick the unpromoted row with the largest `low` and print its challenger, mean gain, interval (6 decimals for `low`) and outcome, plus the two halves of its switching cutoff. Find the promoted row and print its gain and interval, and the drop in CV RMSE in dollars from `reg_results` (incumbent minus champion, times 100,000). Define `rmse_reduction(r2_level, gain) = 1e5 * (sqrt(v*(1-r2_level)) - sqrt(v*(1-r2_level-gain)))` with `v = y_train_reg.var()`, and print it at `champ_reg_mean` for each margin in `DELTAS_R2`, for the rival's `low`, and for the rival's mean gain. Finally print CV R² mean and SD, RMSE and MAE in dollars, and fit time for the champion and the rival."
>
> **After running, verify:**
> - [ ] One line per margin with the champion it implies, and a robust / margin-sensitive line
> - [ ] The closest rival's interval and switching cutoff are printed
> - [ ] The promotion's gain, its interval, and the drop in CV RMSE in dollars are printed
> - [ ] The three margins and the rival's lower limit are printed in dollars of typical error at the champion's level

In [ ]:
# YOUR SOLUTION CODE HERE
# HomeValue: check the champion from §5.2 the way the deployment council will.
#   1. Run the band: champion_ladder(reg_scores, LADDER_REG, delta) for each delta in DELTAS_R2 (robust?).
#   2. Closest rival: the unpromoted challenger with the highest lower limit in ladder_log_reg; print its
#      interval [L, U] and its switching cutoff (worthwhile only below L; equivalent only above max(|L|, |U|)).
#   3. Dollars: translate the champion's promotion into typical error with the CV RMSE of the incumbent
#      and of the champion (reg_results['rmse_mean'] is in units of $100,000), and translate the rival's
#      lower limit L into dollars of typical error at the champion's level:
#      1e5 * (sqrt(v*(1 - R2)) - sqrt(v*(1 - R2 - L))), with v = y_train_reg.var() and R2 = champion's CV mean.
#   4. Checks: CV RMSE and MAE in dollars and fit time for the champion and the closest rival.
#
# Variables available: reg_scores, LADDER_REG, DELTAS_R2, DELTA_R2, ladder_log_reg, champ_reg_name,
#   champ_reg_folds, champ_reg_mean, reg_results, y_train_reg, compare_paired, champion_ladder

### YOUR CHAMPION STATEMENT — HomeValue

Replace each bracket. The **Evidence** line reports the promotion that made the champion; the **Checks** line reports the band and the closest rival with its switching cutoff, in dollars where you can.

> **Champion.** We recommend **[model]** for **[stakeholder]'s [decision]**.
> **Evidence.** Against **[the incumbent]**, on the same five folds, it **[outcome]**: mean gain **[M] R²** (about **[dollars]** less typical error per tract), interval corrected for fold overlap **[L, U]**, against the **δ = [value]** we set before comparing.
> **Checks.** [the small and large margins; the closest rival and its switching cutoff; RMSE and MAE in dollars; fit time]
> **What would change our mind.** **[runner-up]** is the candidate for a future evaluation (not an automatic fallback); we revisit if **[trigger]**.

## 6. Open the Locked Test Set — Once per Case, for the Frozen Champion

§5.2 froze the champions and the exercises checked them. §6 answers the deployment question every stakeholder asks — **"How well should I expect this model to perform on genuinely unseen data, and does that clear the bar we agreed on?"** — in two steps, and nothing in either step selects a model.

### Step 1 — Refit the frozen champion on all of `X_train`

During cross-validation each fold model trained on four-fifths of `X_train`. The model that ships should learn from every development row there is, so the frozen champion is refit on **all of `X_train`** — the 455 patients and the 16,512 tracts. Since nb09 there is no validation set to merge back in: `X_train` *is* the whole development pool.

A small, predictable consequence: the deployed model trains on a quarter more rows than any fold model did, so its test score can come out a little above the CV mean. That is not a bug — cross-validation was measuring slightly smaller models than the one you ship.

### Step 2 — Open the locked test set ONCE and judge it against the deployment bar

Predict on `X_test_*`, compute the primary metric once, and read two lines:

- **The ship decision** — the test score against the **deployment bar declared in §4** (MedScreen ROC-AUC ≥ 0.95; HomeValue $R^2$ ≥ 0.75). Cleared → ship. Missed → **"no model approved under this protocol"**: the champion keeps its name and its failed status, and the runner-up is *not* tested or shipped as a fallback — that would turn the test set into a second selection round.
- **The description** — **INSIDE**, **ABOVE**, or **BELOW** the band *CV mean ± δ*: shorthand for "close to / better than / worse than what cross-validation suggested". It never makes the ship decision, and it never re-picks the champion. §6.3 carries the playbook for each.

### Why NOT score every candidate on the test set and pick the best?

It is tempting to skip the ladder and just run every candidate through the test set:

```
Candidate Model 1 → test performance
Candidate Model 2 → test performance
Candidate Model 3 → test performance
Candidate Model 4 → test performance
Candidate Model 5 → test performance
Choose the best test-set result.
```

**Do not do this.** Picking the best of N test scores **turns the test set into another validation set** — the chosen model was indirectly selected *because it looked good on the test sample*. The winner's reported test score then becomes **optimistically biased**: on a fresh dataset the score regresses back toward the mean of the candidates, often substantially. The whole point of the locked test set is to give an **unbiased** estimate of deployment performance; the moment any candidate's test score influences which model ships, the test set has done two jobs (selection + evaluation), and **neither estimate is honest anymore**.

The discipline is binary: **§5's ladder picks and freezes the champion**; **§6 scores that frozen champion ONCE on test**. The two activities never mix.

> **A question that often comes up here:** *"why refit at all — can't I just score one of the five fold models on the test set?"* Because none of the fold models is the model that ships: each saw only four-fifths of `X_train`, and the five of them are slightly different models. The stakeholder will deploy one model trained on every development row available, so that is the model whose test score belongs in the approval package. The cost of the refit is one `fit(...)` call before the test prediction; the cost of skipping it is a headline number that does not describe the production model.

### 6.1 Classification Ceremony — Wisconsin Breast Cancer

The MedScreen ceremony evaluates the frozen champion, `LogReg C=1`, against `X_test_clf` — the 114 patients held out since nb06. **Step 1** refits it on all 455 training patients. **Step 2** opens the locked envelope for the one and only authorized prediction pass, computes test ROC-AUC (primary), accuracy, and F1, checks the 0.95 deployment bar, and describes where the test score landed relative to the band CV mean ± 0.01.

The money plot puts the pieces side by side: the champion's five CV fold scores (gray dots), the CV mean ± δ band (gray, a description), the deployment bar (red dashed line, the ship decision), and the one test score (the triangle — green if it clears the bar).

> **A question that often comes up here:** *"who at MedScreen actually sees the test-set number, and what do they do with it?"* The State Health Department's review board sees it as the headline metric in the approval package — *"on a held-out set of 114 patients the screening model achieved a test ROC-AUC of X, above the 0.95 bar declared in advance."* The board does not run the model itself; it reads the number, compares it to the bar and to the screening protocol it replaces, and decides whether to authorize the rollout. The INSIDE / ABOVE / BELOW description tells the board whether the test score looks like what cross-validation suggested; the authorization rests on the bar, and a missed bar sends the package back with "no model approved under this protocol" — not with the runner-up stapled on.

> 💡 **Gemini Prompt:** "Open the LOCKED classification test set EXACTLY ONCE, for the champion frozen in §5.2 — do not select or re-rank anything in this cell. Assert that `champ_clf_name` exists. **Step 1 — refit**: `champion_clf = clone(clf_models[champ_clf_name])`, fit it on all of `(X_train_clf, y_train_clf)`. **Step 2 — score on test**: predict probabilities and classes on `X_test_clf` and compute test ROC-AUC, accuracy, and F1. Set `ship_clf = test_auc >= DEPLOY_BAR_CLF` and `where_clf = describe_test(test_auc, champ_clf_mean, DELTA_AUC)`. Print a header block with the champion name, its five CV fold scores and mean, the refit size, the test scores, the deployment-bar line (CLEARED — ship / MISSED — no model approved under this protocol), and the description line. Then draw the **money plot**: a gray band from `champ_clf_mean - DELTA_AUC` to `champ_clf_mean + DELTA_AUC` labelled 'description only', the five fold scores as gray dots, a gray line at the CV mean, a red dashed line at `DEPLOY_BAR_CLF` labelled as the ship decision, and the test score as a triangle (green if it clears the bar, red otherwise)."
>
> **After running, verify:**
> - [ ] The cell selects nothing: it uses `champ_clf_name` from §5.2
> - [ ] Step 1 refits on all of `X_train_clf` (455 patients)
> - [ ] The header prints the test ROC-AUC / accuracy / F1, the deployment-bar line, and the INSIDE / ABOVE / BELOW description
> - [ ] The money plot shows the fold scores, the CV mean ± δ band, the dashed deployment bar, and the test triangle

In [ ]:
# CEREMONY — CLASSIFICATION
# X_test_clf opens here, exactly once, for the champion FROZEN in §5.2. Nothing in this cell selects a model.
assert 'champ_clf_name' in globals(), "Run §5.2 first: the test set opens only for a frozen champion."

# Step 1 — REFIT the frozen champion on all of X_train (the 455 patients the five folds came from).
champion_clf = clone(clf_models[champ_clf_name])
champion_clf.fit(X_train_clf, y_train_clf)

# Step 2 — SCORE on the locked test set, exactly ONCE.
y_proba_test_clf = champion_clf.predict_proba(X_test_clf)[:, 1]
y_pred_test_clf  = champion_clf.predict(X_test_clf)
test_auc      = roc_auc_score(y_test_clf, y_proba_test_clf)
test_accuracy = accuracy_score(y_test_clf, y_pred_test_clf)
test_f1       = f1_score(y_test_clf, y_pred_test_clf)

ship_clf  = test_auc >= DEPLOY_BAR_CLF                                # the ship decision
where_clf = describe_test(test_auc, champ_clf_mean, DELTA_AUC)        # a description only

print('━' * 60)
print('CEREMONY — CLASSIFICATION (MedScreen, Wisconsin Breast Cancer)')
print('━' * 60)
print(f'Frozen champion:     {champ_clf_name}')
print(f'CV ROC-AUC folds:    {np.round(champ_clf_folds, 4)}  (mean {champ_clf_mean:.4f})')
print(f'Refit on:            X_train, {len(X_train_clf)} patients')
print(f'Test ROC-AUC:        {test_auc:.4f}   (accuracy {test_accuracy:.4f}, F1 {test_f1:.4f})')
print(f'Deployment bar:      test ROC-AUC ≥ {DEPLOY_BAR_CLF:.2f} → '
      f'{"CLEARED — ship" if ship_clf else "MISSED — no model approved under this protocol; diagnose"}')
print(f'Description:         {where_clf} the band CV mean ± δ = [{champ_clf_mean - DELTA_AUC:.4f}, {champ_clf_mean + DELTA_AUC:.4f}]')

# The money plot: the champion's five fold scores, the CV mean ± δ band, the bar, and the one test score
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.axvspan(champ_clf_mean - DELTA_AUC, champ_clf_mean + DELTA_AUC, color=GREY, alpha=0.2,
           label=f'CV mean ± δ (description only): {champ_clf_mean:.4f} ± {DELTA_AUC}')
ax.scatter(champ_clf_folds, np.zeros(5), color=GREY, s=60, zorder=3, label='The champion\'s five CV fold scores')
ax.axvline(champ_clf_mean, color=GREY, linewidth=2)
ax.axvline(DEPLOY_BAR_CLF, color=RED, linestyle='--', linewidth=2, label=f'Deployment bar {DEPLOY_BAR_CLF} (the ship decision)')
ax.scatter([test_auc], [0], color=GREEN if ship_clf else RED, marker='^', s=300, zorder=5,
           label=f'Test score {test_auc:.4f} → {"ship" if ship_clf else "do not ship"} ({where_clf})')
ax.set_yticks([]); ax.set_xlabel('ROC-AUC')
ax.set_title(f'Classification ceremony — {champ_clf_name}', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

**Reading the output:**

The frozen champion, refit on all 455 training patients, scores a **test ROC-AUC of 0.9993** on the 114 locked patients (accuracy 0.9737, F1 0.9790). That clears the **0.95 deployment bar** declared in §4, so the ship line prints **CLEARED — ship**. The description line prints **INSIDE**: the test score sits within ±0.01 of the CV mean of 0.9941, a little above it.

Read the two lines in that order. The bar is the decision; the description is a sanity check. INSIDE is reassuring — nothing surprising happened between development and the envelope — but it is not proof that the model generalizes: the test score is one number computed on 114 patients, and the band around the CV mean is a margin the board chose, not a statistical guarantee. A test score above the CV mean would not be alarming either, for the reason Step 1 gave: the shipped model learned from all 455 patients, while every fold model learned from 364.

**The classification test set is now closed. It will not reopen in this course.**

> **A question that often comes up here:** *"what do I say to MedScreen's review board if the description had come out ABOVE?"* That a modest ABOVE is expected here and why: the deployed model trained on 455 patients while each fold model trained on 364 — a quarter more data than cross-validation measured — and a 114-patient test set moves around more than a 4,000-row one. Write that explanation down before the meeting. What you never do with an ABOVE (or any) description is change the champion or the bar: the ship decision was the bar, and it was declared before the envelope opened. A score far above the band is worth a short investigation, which §6.3 describes; a score near it is good news, stated honestly.

---

### 6.2 Regression Ceremony — California Housing

Same two-step protocol on the regression side. **Step 1**: refit the frozen champion — the random forest with scikit-learn's defaults — on all 16,512 training tracts. **Step 2**: open the locked test set `X_test_reg` for exactly one prediction pass, check the 0.75 $R^2$ deployment bar, translate the errors into dollars, and describe where the test score landed relative to the band CV mean ± 0.01.

The tuned GBM does **not** enter this cell. It had the higher CV mean, but its gain over the forest was unresolved against the CFO's margin, so the ladder kept the forest — and the ceremony opens the envelope for the frozen champion only.

> **A question that often comes up here:** *"isn't shipping the forest leaving accuracy on the table, when the tuned GBM had the higher CV mean?"* Possibly — and the honest report says so. The GBM's gain over the forest, +0.0181 $R^2$ with an interval from +0.0091 to +0.0271, is consistent with a gain the CFO would act on and with one just short of her margin; her policy, declared before scoring, keeps the champion unless the challenger shows a worthwhile gain, and Exercise 2 showed that a CFO with a smaller margin would have switched. If she wants to bet on the GBM, that is a legitimate business call — made explicitly and on the record, with the switching cutoff in hand — but it is not something the test set may settle: scoring both models on the envelope and keeping the better one would be exactly the selection-by-test-set this section forbids. The GBM stays on the record as the candidate for the next evaluation, not as a fallback.

> 💡 **Gemini Prompt:** "Open the LOCKED regression test set EXACTLY ONCE, for the champion frozen in §5.2 — do not select or re-rank anything in this cell. Assert that `champ_reg_name` exists. **Step 1 — refit**: `champion_reg = clone(reg_models[champ_reg_name])`, fit on all of `(X_train_reg, y_train_reg)`. **Step 2 — score on test**: predict on `X_test_reg` and compute test R², RMSE, and MAE; multiply RMSE and MAE by 100,000 to translate California Housing's target units into dollars. Set `ship_reg = test_r2 >= DEPLOY_BAR_REG` and `where_reg = describe_test(test_r2, champ_reg_mean, DELTA_R2)`. Print a header block with the champion name, its five CV fold scores and mean, the refit size, the test R² / RMSE / MAE (raw and in dollars), the deployment-bar line, and the description line. Draw the same money plot as §6.1 for R²."
>
> **After running, verify:**
> - [ ] The cell selects nothing: it uses `champ_reg_name` from §5.2
> - [ ] Step 1 refits on all of `X_train_reg` (16,512 tracts)
> - [ ] `X_test_reg` opens only in this cell
> - [ ] The header prints the test R² / RMSE / MAE (raw and in dollars), the deployment-bar line, and the INSIDE / ABOVE / BELOW description

In [ ]:
# CEREMONY — REGRESSION
# X_test_reg opens here, exactly once, for the champion FROZEN in §5.2. Nothing in this cell selects a model.
assert 'champ_reg_name' in globals(), "Run §5.2 first: the test set opens only for a frozen champion."

# Step 1 — REFIT the frozen champion on all of X_train (the 16,512 tracts the five folds came from).
champion_reg = clone(reg_models[champ_reg_name])
champion_reg.fit(X_train_reg, y_train_reg)

# Step 2 — SCORE on the locked test set, exactly ONCE.
y_pred_test_reg = champion_reg.predict(X_test_reg)
test_r2   = r2_score(y_test_reg, y_pred_test_reg)
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_test_reg))
test_mae  = mean_absolute_error(y_test_reg, y_pred_test_reg)

ship_reg  = test_r2 >= DEPLOY_BAR_REG                                 # the ship decision
where_reg = describe_test(test_r2, champ_reg_mean, DELTA_R2)          # a description only

print('━' * 60)
print('CEREMONY — REGRESSION (HomeValue, California Housing)')
print('━' * 60)
print(f'Frozen champion:     {champ_reg_name}')
print(f'CV R² folds:         {np.round(champ_reg_folds, 4)}  (mean {champ_reg_mean:.4f})')
print(f'Refit on:            X_train, {len(X_train_reg):,} tracts')
print(f'Test R²:             {test_r2:.4f}')
print(f'Test RMSE:           {test_rmse:.4f}  (USD {test_rmse * 100_000:,.0f})')
print(f'Test MAE:            {test_mae:.4f}  (USD {test_mae * 100_000:,.0f})')
print(f'Deployment bar:      test R² ≥ {DEPLOY_BAR_REG:.2f} → '
      f'{"CLEARED — ship" if ship_reg else "MISSED — no model approved under this protocol; diagnose"}')
print(f'Description:         {where_reg} the band CV mean ± δ = [{champ_reg_mean - DELTA_R2:.4f}, {champ_reg_mean + DELTA_R2:.4f}]')

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.axvspan(champ_reg_mean - DELTA_R2, champ_reg_mean + DELTA_R2, color=GREY, alpha=0.2,
           label=f'CV mean ± δ (description only): {champ_reg_mean:.4f} ± {DELTA_R2}')
ax.scatter(champ_reg_folds, np.zeros(5), color=GREY, s=60, zorder=3, label='The champion\'s five CV fold scores')
ax.axvline(champ_reg_mean, color=GREY, linewidth=2)
ax.axvline(DEPLOY_BAR_REG, color=RED, linestyle='--', linewidth=2, label=f'Deployment bar {DEPLOY_BAR_REG} (the ship decision)')
ax.scatter([test_r2], [0], color=GREEN if ship_reg else RED, marker='^', s=300, zorder=5,
           label=f'Test score {test_r2:.4f} → {"ship" if ship_reg else "do not ship"} ({where_reg})')
ax.set_yticks([]); ax.set_xlabel('R²')
ax.set_title(f'Regression ceremony — {champ_reg_name}', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

**Reading the output:**

The frozen forest, refit on all 16,512 training tracts, scores a **test $R^2$ of 0.8005** on the 4,128 locked tracts — a typical error (RMSE) of about **\$51,800** and a mean absolute error of about **\$33,700** per tract. That clears the **0.75 deployment bar**, so the ship line prints **CLEARED — ship**, and the description is **INSIDE**: the test score sits within ±0.01 of the CV mean of 0.8036, a little below it.

The money plot tells the same story at a glance: the triangle sits among the five gray fold scores, well to the right of the dashed bar. Read it as one honest observation placed next to the cross-validation evidence, and read the ship decision off the bar line.

Translating $R^2$ into dollars is the version stakeholders will quote. *"$R^2$ = 0.81"* is abstract; *"the model's typical miss on a tract's median home value is about \$52,000"* is a number HomeValue's deployment council can defend or contest in concrete terms.

**The regression test set is now closed. It will not reopen in this course.**

> **A question that often comes up here:** *"how should HomeValue decide when to retrain the model after deployment?"* Two triggers, both declared now and written on the monitoring card in §7.5. (1) **Time-based**: every Q1, retrain on the latest year of housing data and re-run the §4–§6 protocol on a fresh roster and a fresh test set. (2) **Drift-based**: if the production median prediction error (MAE in dollars) on the most recent 30 days of closed sales drifts beyond about 1.5× the cross-validated MAE recorded at deployment, fire a retrain immediately and re-open the protocol. The discipline is the same one nb14 enforces — never let test-set evidence influence the next model's selection without the protocol being declared first. The deployment council does not have to remember the test-set number forever; it has to remember the rule that fires the next ceremony.

---

### 6.3 Reading the Ceremony — the Ship Decision, Then the Description

Every ceremony prints two lines. The **deployment-bar line decides**: cleared → ship; missed → *no model approved under this protocol*. The **INSIDE / ABOVE / BELOW line describes** where the test score landed relative to the champion's CV mean ± δ, and each description comes with a short playbook of what to record and what to look into. None of them ever re-picks the champion.

**INSIDE — within ± δ of the CV mean.** The deployed model scored close to what the cross-validation evidence suggested; nothing surprising happened between development and the envelope. **Ship if the bar is cleared.** The deployment memo records (1) the model class and hyperparameters, (2) the champion statement with its CV evidence — fold scores, mean, SD, and the ladder log, (3) the test score, the bar it cleared, and the date the envelope opened, and (4) the retrain triggers. INSIDE is the description you hope for — read it as the absence of a nasty surprise rather than as proof that the model will generalize.

**ABOVE — more than δ above the CV mean.** Two cases with different actions. A *modest* ABOVE is unremarkable: the shipped model trained on all of `X_train`, a quarter more rows than any fold model, so cross-validation was measuring slightly weaker models. **Ship if the bar is cleared and document the gap** so the stakeholder understands why the headline number is higher than the development estimate. A *large* ABOVE — several margins above the CV mean — deserves a look before deploying. Three possible causes, in order of likelihood: (1) an easy test draw, rare on California Housing's 4,128 tracts but plausible on MedScreen's 114 patients; (2) something that depresses the CV scores but not the test score, such as a derived feature computed differently inside the folds; (3) a data-quality issue isolated to the training rows — mislabeled rows, outliers, a collection-window difference between train and test. Document the investigation; never "fix" a large ABOVE by switching models.

**BELOW — more than δ below the CV mean.** Treat it as the ceremony's warning light: the cross-validation evidence painted a rosier picture than the one honest observation. **If the score also misses the bar, the outcome is "no model approved under this protocol": do not deploy, and do not reach for the runner-up.** Even when the bar is cleared, diagnose before shipping. Three diagnoses in order of likelihood:

1. **Leakage during development.** Some feature engineering or preprocessing happened *outside* the folds, leaking information from held-out rows into training rows. Audit every pipeline step: target encoding, scaling, imputation, derived features. The rule is *"everything fits inside the fold, nothing fits outside it."* If you find leakage, fix the pipeline, re-run the protocol, and **open a NEW held-out test set** — the old one is now contaminated by the result you just saw.
2. **Distribution shift.** The test rows differ systematically from the training rows — a different time period, sampling frame, or label noise. Compare summary statistics between `X_train` and `X_test` feature by feature. If shift is confirmed, the right response is *not* to retrain on the test set; it is to refresh the development data so both reflect the deployment distribution, then re-run the chain from nb09 forward.
3. **Insufficient training signal.** Too few examples per class on classification, too few rows in a key region of feature space on regression. The fix is more data, not a different model.

The hardest rule is the **test-set-is-spent** principle: opening the test set once consumes its credibility, whatever the result. If you diagnose, refit, and want to score again, the second score needs a **new** test set held out from the start. Reusing the same test set would turn the envelope into a validation set — exactly the failure mode §6's ceremony exists to prevent.

> **A question that often comes up here:** *"if a missed bar means I can't reuse the test set, am I stuck waiting for new data to retry?"* Often yes — and that is the right discipline. In professional practice the team holds out a *gold* test set at the very start of the project that stays sealed until the ceremony, AND keeps a *deployment monitoring* slice sampled fresh from production that is never used for selection. If the ceremony fails, the gold set has done its job by telling you so; the monitoring slice catches drift after release. The test-set discipline is not about being precious with data — it is about preserving the one mechanism that can give an unbiased estimate of deployment performance, which is the only number stakeholders will treat as binding.

---

## 7. Moving Ahead — Keeping the Model Honest After You Ship It

The ceremony in §6 was a one-time event: you opened the locked test set, wrote down the verdict, and closed it for good. But in real life the model keeps making predictions long after that day — new patients arrive for MedScreen, new properties get listed for HomeValue — and the world the model lives in keeps changing. A deployed model **decays** quietly. The accuracy you measured today is not guaranteed to hold next quarter, and the only way to know is to **watch the model in production**.

This section is about that watching. §6 named *which model ships*. §7 answers two follow-up questions every stakeholder will eventually ask: *"how do I know the model is still working?"* and *"when do I do something about it?"*

**The production world can change in two very different ways**, and each calls for a different response. We will call them **Silent decay** (changes that hit *without* announcement — only monitoring catches them) and **Front-door arrivals** (changes that show up *with* notice — your job is a disciplined decision, not detection).

**Silent decay — the quiet quality slip.** Three failure modes that hit a deployed model without warning. The model keeps returning predictions, nothing crashes, but quality slips. These are invisible unless you set up alerts.

1. **Data drift** *(also called covariate shift).* The inputs change — same column names, different values. MedScreen buys new microscopes that measure cell size slightly differently. HomeValue's `MedInc` numbers shift as inflation pushes incomes up. The model never sees the change explicitly; its predictions just get less accurate.

2. **Concept drift.** The *relationship* between inputs and the answer changes. A neighborhood gentrifies and high incomes no longer mean what they meant a few years ago. A new biopsy staining technique exposes a tumor marker the model never trained on, so the old features stop carrying the predictive signal they used to. Same numbers, different meaning.

3. **Population shift.** The people (or properties) the model is applied to change. MedScreen extends screening eligibility from women 50+ to women 40+. HomeValue is asked to price vacation rentals — a property type the training data did not include. The model is now being used on a population it was never trained on.

**Front-door arrivals — the announced change.** The other category is different in tone. Instead of quality slipping in the dark, *more signal* arrives in plain view: a NEW feature becomes available, and somebody tells you about it. Six months after deployment, MedScreen's lab adds a new tissue-marker test to every biopsy panel, or HomeValue gets access to a new census variable nobody had access to before. This is a *good* problem — extra information showing up, not quality dropping — but the response is still disciplined, because the wrong decision can hurt you.

The right response depends on what kind of new feature it is:

- **One new column on the same kind of row** — for example, a new lab measurement attached to each biopsy, or one extra census variable on each property. **Plug the new column into the existing pipeline and refit the same model class with the bigger feature set.** Only put the new version into production if it earns the job the way every challenger in §5.2 had to: score both versions on the identical five folds and promote the new one only when `compare_paired(new, deployed, δ)` says *A is better by a worthwhile amount*.

- **A feature that changes the kind of data you have** — for example, a measurement that needs entirely new preprocessing (a new encoder, a new way of handling missing values), or one that suggests a fundamentally different model family would now work. **Re-open the nb14 pipeline (§§4–6) with the new feature included from the start** — a fresh candidate roster, fresh cross-validation, a new champion memo, and a new ceremony to open a fresh test set.

**Three things to check before plugging *any* new feature in**, regardless of which response you choose:

- **Could the new feature be leaking?** A *leaky* feature is one that secretly contains information about the answer. (Example: a "doctor's clinical note" field that was filled in *after* the biopsy result was known will look incredibly predictive — because the note literally summarizes the answer.) The four-method importance heatmap from nb12 §7 is the right diagnostic: if a brand-new feature suddenly dominates all four importance methods and the training error drops dramatically, treat it as a leakage suspect until you can prove it isn't. Gradient boosting (nb13) is especially vulnerable here — its very design hunts for the most informative feature first.

- **Does the new feature actually help by a meaningful margin?** Apply the same comparison as §5.2. Score the with-feature and without-feature models on the identical folds and call `compare_paired(with_feature, without_feature, δ)` with the margin you declared in advance. Only *A is better by a worthwhile amount* ships the bigger model; practically equivalent, unresolved, or worse — the deployed model stays, because you do not deploy more complexity unless the evidence shows a gain worth having.

- **What happens in production when the new feature is missing?** Real-world rows will sometimes lack the new column (the new lab test was not ordered for this patient; the new census release is not yet available for this tract). Decide in advance: do you fill in a sensible default (impute), fall back to the deployed model that did not need the feature, or refuse to predict on those rows? Make the call before you ship.

**Most of the rest of §7 is about catching Silent decay**, because that is the harder problem — you don't get a phone call. The next subsection (§7.1) lays out the *when to act* policy: an escalation ladder of three possible responses, ranked from cheapest to most expensive. §7.2 and §7.3 give the per-case monitoring checklists; §7.4 walks one drift-detection technique in code; §7.5 collapses everything onto a one-page monitoring card. The Front-door-arrival workflow above and the Silent-decay workflow share the same escalation framework, which §7.1 introduces next.

---

### 7.1 Three Things You Can Do — Re-evaluate, Re-train, Rebuild

When monitoring spots something off, you have three possible actions. They come at very different costs, so the rule is: **always reach for the cheapest one that fixes the problem.** Think of it as a three-step ladder where you start at the bottom and only climb if the lower step does not solve it.

1. **Step 1 — Re-evaluate.** *Low cost — a few minutes against a fresh CSV.* Take the exact model already running in production. Score it on the most recent batch of new data (where you already know the right answers). No refitting, no model changes — just *"is the model that has been running still accurate on what just happened?"* Fires on routine monthly or quarterly checks, or whenever monitoring flags something that might be drift.

2. **Step 2 — Re-train.** *Medium cost — same compute as the original training run.* Same model class, same algorithm, same hyperparameters — but **refit on fresh data** that includes the months you have collected since deployment. Compare the refit against the deployed model with `compare_paired(refit, deployed, δ)` on identical folds of the fresh data, and promote the refit only when it is better by a worthwhile amount; in every other outcome the incumbent keeps running. Fires when Step 1 confirms a real drop, when the data window has grown enough to matter, or when a production metric has crossed the **business alert threshold declared on the monitoring card** (§7.5) for **two monitoring periods in a row**.

3. **Step 3 — Rebuild.** *High cost — roughly the same effort as running nb14 from scratch.* Re-open the full nb14 pipeline (§§4–6): a fresh candidate roster (maybe with new model families nobody tried originally), fresh cross-validation, a new champion memo, and a new ceremony to open a fresh test set. Fires when Step 2's refit still cannot bring performance back to an acceptable level, OR when something fundamental changed (a new feature is now available, the business problem itself shifted, the deployed model's lift over the linear baseline has narrowed too much).

**The escalation rule is mechanical.** Always start at Step 1. If Step 1 confirms drift, escalate to Step 2. If Step 2's refit cannot earn a worthwhile gain or restore acceptable performance, only then escalate to Step 3. The reason for the cheapest-first discipline is simple: **Step 3 is essentially a full re-do of nb14**. It costs real time and real money. Make stakeholders sign off before you go up that step.

---

### 7.2 MedScreen Monitoring Plan — What to Track for the Classification Model

For the State Health Department's screening tool, MedScreen ships `LogisticRegression(C=1.0)` (the §6 champion). The model outputs probabilities, and the board will read them as probabilities — the model's *"73%"* should mean *"about 73% of patients with this score turn out to be malignant."* Whether it does is something to **check, not assume**: logistic regression's probabilities are often reasonably close to the observed frequencies, but no model family guarantees it, and drift can break it after deployment. The good news is that the check is direct: compare the probabilities the model puts out against what actually happened.

The first item in the checklist below uses two tools to make that check concrete:

- The **reliability diagram** is a chart that compares *"the model said 70%"* against *"how often the answer actually was malignant when the model said 70%."* A well-calibrated model's curve hugs the 45° diagonal — predicted 70%, observed 70%.
- The **Brier score** is a single number that summarizes the reliability diagram — *"on average, how far off were the predicted probabilities from the actual outcomes?"* Smaller is better, and you compare it against the cross-validated Brier score recorded at deployment (Exercise 1 computed it: 0.0247).

The other priority on a screening tool is **recall** (also called sensitivity) — what fraction of actual malignancies the model catches. A screening model that misses real malignancies is much worse than one that occasionally over-flags a benign case, so recall is the number to watch most closely.

**MedScreen monitoring checklist** (five items, each with cadence, alert threshold, and the right escalation step):

1. **Reliability diagram + Brier score** *(on the last 200 confirmed biopsies)*
   - **How often:** monthly
   - **Flag when:** the Brier score climbs above 1.5× the cross-validated value recorded at deployment, OR the reliability curve no longer hugs the diagonal
   - **Action:** Step 1 (Re-evaluate)

2. **Recall** *(at the threshold currently used to flag a biopsy for confirmatory testing)*
   - **How often:** monthly
   - **Flag when:** recall drops below 0.92 over a rolling 90-day window
   - **Action:** Step 2 (Re-train), refitting on the most recent 24 months of biopsy data

3. **Input drift** *(on the top-3 features the model relies on — `worst concave points`, `worst perimeter`, `worst radius`)*
   - **How often:** quarterly
   - **Flag when:** a drift test (§7.4 walks the KS option and points at alternatives) crosses your alert threshold
   - **Action:** Step 1 first; if drift is confirmed, escalate to Step 2

4. **Class prevalence** *(how often the answer is actually malignant)*
   - **How often:** quarterly
   - **Flag when:** the malignant rate has shifted by more than ±20% from what the training data showed
   - **Action:** Step 1; you may also need to move the threshold to keep the model useful at the new base rate

5. **Annual clinical review** *(by the State Health Department's review board)*
   - **How often:** yearly
   - **Flag when:** new screening guideline, new imaging equipment, OR a new tissue-marker test becomes available
   - **Action:** Step 3 (Rebuild) — or Step 2 Re-train if the new marker is just an extra column (see §7's opening on new features)

**Who owns this?** The data steward who maintains the biopsy data pipeline runs the monthly checks. The oncologist co-signs every quarterly review. Both sign off jointly before any Step-2 or Step-3 action goes live. In practice, the monitoring checklist above becomes a **one-page report** the State Health Department's review board reads at the start of every quarterly meeting.

---

### 7.3 HomeValue Monitoring Plan — What to Track for the Regression Model

For HomeValue Analytics' price-prediction tool, the champion is the random forest with scikit-learn's defaults, `RandomForestRegressor()` — promoted over OLS in §5.2 by a worthwhile gain, kept against the tuned GBM (unresolved at the CFO's base margin, a margin-sensitive call), and confirmed at the §6.2 ceremony. Unlike MedScreen's logistic regression, the forest does not output probabilities — it outputs **dollar amounts**. So the monitoring story shifts to two questions: *how big are the prediction errors right now*, and *do those errors get worse for certain kinds of properties* (for example, the most expensive tracts, where errors are largest in dollars).

**HomeValue monitoring checklist** (six items, each with cadence, alert threshold, and the right escalation step):

1. **Average and typical prediction errors (MAE / RMSE)** *(on last month's appraisals)*
   - **How often:** monthly
   - **Flag when:** RMSE climbs above USD 70,000 (compared to the \~USD 51,000 the forest showed in §5's cross-validation) over a rolling 90-day window
   - **Action:** Step 1 (Re-evaluate)

2. **Errors broken out by price tier** *(split properties into five equal-sized price bins — quintiles — and compute RMSE in each)*
   - **How often:** quarterly
   - **Flag when:** top-quintile (the priciest 20% of properties) RMSE climbs more than 25% above the top-quintile RMSE recorded on the monitoring card at deployment
   - **Action:** Step 2 (Re-train) on the most recent 36 months of sales

3. **Errors by location** *(group properties into a coarse latitude/longitude grid; compute average error in each cell with more than 100 properties)*
   - **How often:** quarterly
   - **Flag when:** any cell's average error has shifted by more than half a standard deviation from what it was at training time
   - **Action:** Step 1 on that cell; consider a region-specific refit if a neighborhood has genuinely changed

4. **Input drift** *(on the most important features — `MedInc`, `AveOccup`, `Population`)*
   - **How often:** monthly
   - **Flag when:** a drift test (§7.4 walks the KS option and points at alternatives) crosses your alert threshold
   - **Action:** Step 1; if confirmed, escalate to Step 2

5. **Major outside events**
   - **How often:** whenever one happens (event-driven)
   - **Flag when:** the Federal Reserve changes interest rates by more than 50 basis points (half a percentage point), OR the US Census releases a new wave of demographic data, OR a major housing-policy change takes effect
   - **Action:** Step 1 immediately, and flag the deployment council that the model may need attention

6. **Annual model audit** *(by the deployment council)*
   - **How often:** yearly
   - **Flag when:** the deployed forest's advantage over the simple OLS baseline has narrowed to fewer than 10 R² points, OR a new dataset becomes available with previously-unavailable features
   - **Action:** Step 3 (Rebuild) — or Step 2 Re-train if the new feature is just an extra column (see §7's opening on new features)

**Who owns this?** The pricing-analytics team runs the monthly checks. The deployment council co-signs the quarterly review. The "major outside events" item is set up to fire automatically off public-data feeds (Federal Reserve announcements, Census releases, and so on) — the data steward keeps that script running. Same one-page-report discipline as MedScreen.

---

### 7.4 Drift Detection in Code — One Approach (and a Note That Others Exist)

The monitoring checklists above kept using the phrase *"a drift test"* without explaining what one looks like in code. This subsection walks **one example** — the **Kolmogorov-Smirnov test**, usually abbreviated as the **KS test**. It is one of the simplest drift detectors and a good starting point for the course. **It is not the only option**, and depending on your data and your business setting you may want to research alternatives (see the list at the end of this subsection).

**What does a KS test do, in one sentence?** Take the distribution of a feature in your training data, take the distribution of the same feature in your recent production data, and measure how far apart those two distributions are. If the two distributions look basically the same shape, the test returns a small number (no drift). If they look meaningfully different — shifted left, shifted right, narrower, wider — the test returns a larger number (drift detected). The "KS statistic" itself is a number between 0 and 1 — bigger means more different. In production you would pick a threshold (the monitoring checklists above used 0.10 for HomeValue's features and 0.15 for MedScreen's, but those are starting values you tune for your case after observing some real production data) and treat any feature where the KS statistic exceeds the threshold as an alert.

The code cell below runs this on one important feature from each business case. For MedScreen we use `worst perimeter` (one of the top features the logistic regression relies on). For HomeValue we use `MedInc` (the single most important feature by permutation importance from nb12). For each case we run **three scenarios**: a *no-shift* baseline (the training distribution compared to itself — must always come back `✓ OK`), a *modest* simulated shift that should fall under the 0.10 alert threshold, and a *bigger* simulated shift that should clearly cross it. Then we read the KS test results side by side and visualize the bigger-shift production distributions against training.

> **Other drift-detection options worth researching when you build this for a real system.** The KS test is just one tool in a much wider area called *data-drift detection*. Other widely used approaches include:
>
> - **Population Stability Index (PSI)** — heavily used in banking and credit-risk modeling.
> - **Chi-squared test** — the natural choice for *categorical* features (where the KS test does not work directly).
> - **Wasserstein distance** (also called *Earth Mover's Distance*) — measures the *size* of a shift, not just whether one happened.
> - **Jensen-Shannon divergence** — an information-theoretic measure of how different two distributions are.
> - **Plain summary-statistic comparisons** — comparing means, standard deviations, percentiles side-by-side over time. Often the simplest dashboard is the most useful.
>
> Different industries default to different tools — finance leans on PSI, manufacturing reaches for Wasserstein when small tolerance shifts matter, healthcare often uses chi-squared on categorical lab-result fields. The right choice depends on your data and what kind of shift you most want to catch. A quick search for *"data drift detection methods"* will turn up active research and several open-source libraries (`evidently`, `alibi-detect`, `nannyml` are three popular ones) that bundle multiple tests into a single workflow. None of these are required for the work this notebook covers — they are starter pointers for when your group is ready to build monitoring for a real deployed model.

> 💡 **Gemini Prompt:** "Write a Python cell that runs a Kolmogorov-Smirnov drift test on one feature from each business case. Use `scipy.stats.ks_2samp`. Define `drift_check(name, train_values, prod_values, alert_threshold=0.10)` that returns a formatted string starting with `'🚨 DRIFT'` when the KS statistic exceeds `alert_threshold` and `'✓ OK'` otherwise, followed by the feature name (left-padded to 40 chars), the KS statistic to 3 decimals, and the p-value to 3 significant figures.
>
> For the **classification** case, pull `X_train_clf['worst perimeter']` and create three production scenarios drawn from one `np.random.RandomState(RANDOM_SEED)`: (a) the training values unchanged ('no shift'), (b) a *small sensor drift* adding `normal(0, 5, n)` noise, and (c) a *bigger sensor drift* adding `normal(0, 20, n)` noise — both drawn sequentially from the same RNG so the runs are reproducible. Print all three rows under a `=== CLASSIFICATION (MedScreen) ===` header.
>
> For the **regression** case, pull `X_train_reg['MedInc']` and create three scenarios: (a) no shift, (b) a 10% upward income shift (`* 1.10`), and (c) a 20% upward shift (`* 1.20`). Print all three rows under a `=== REGRESSION (HomeValue) ===` header.
>
> Then build a 1×2 figure with `plt.subplots(1, 2, figsize=(15, 5))`. The left panel overlays the MedScreen training distribution against the *std=20* production distribution. The right panel overlays the HomeValue training distribution against the *20%* production distribution. Use `CLF_COLOR` / `REG_COLOR` for training, `RED` for the drifted production. Add 30-bin histograms with `alpha=0.5`, titles, axis labels, legends, gridlines, and a `fig.suptitle` describing what the figure shows."
>
> **After running, verify:**
> - [ ] Six labeled rows total (three per case), printed under their respective `=== CASE ===` headers
> - [ ] Both "no shift" rows show `KS=0.000` and `✓ OK`
> - [ ] The modest-shift rows in both cases come in *below* the 0.10 threshold and show `✓ OK`
> - [ ] The bigger-shift rows in both cases come in *above* the 0.10 threshold and show `🚨 DRIFT`
> - [ ] The 1×2 figure renders with the bigger-shift production histogram visibly offset from training in both panels
> - [ ] The HomeValue p-values are dramatically smaller than the MedScreen p-values, even when the KS statistics are similar — that asymmetry is one of the lessons in the Reading-the-output below

In [ ]:
# §7.4 — Drift detection demo: KS (Kolmogorov-Smirnov) test on one feature per business case.
#
# What the KS test returns:
#   - `stat` is between 0 and 1. Bigger = the two distributions look more different.
#   - `p` is a p-value. Smaller means the two distributions are more confidently different.
# We use only `stat` for our alert decision (a stat above the alert threshold = DRIFT).
# The p-value is informative but should NOT drive the alert by itself — see the Reading
# below for why large sample sizes shrink p-values regardless of how big the shift is.

from scipy.stats import ks_2samp

def drift_check(name, train_values, prod_values, alert_threshold=0.10):
    stat, p = ks_2samp(train_values, prod_values)
    flag = '🚨 DRIFT' if stat > alert_threshold else '✓ OK'
    return f'{flag}  {name:<40} KS={stat:.3f}  p={p:.3g}'

# --- Classification (MedScreen): 'worst perimeter' is a top feature for the LogReg champion ---
clf_train_feat  = X_train_clf['worst perimeter']
rng = np.random.RandomState(RANDOM_SEED)
clf_prod_modest = clf_train_feat + rng.normal(0,  5, len(clf_train_feat))   # small sensor noise
clf_prod_bigger = clf_train_feat + rng.normal(0, 20, len(clf_train_feat))   # heavier sensor noise

print('=== CLASSIFICATION (MedScreen) ===')
print(drift_check('worst perimeter (no shift)',            clf_train_feat, clf_train_feat))
print(drift_check('worst perimeter (small sensor drift)',  clf_train_feat, clf_prod_modest))
print(drift_check('worst perimeter (bigger sensor drift)', clf_train_feat, clf_prod_bigger))

# --- Regression (HomeValue): 'MedInc' is the most important feature by permutation importance from nb12 ---
reg_train_feat  = X_train_reg['MedInc']
reg_prod_modest = reg_train_feat * 1.10    # 10% upward income shift (inflation / gentrification)
reg_prod_bigger = reg_train_feat * 1.20    # 20% upward income shift

print('\n=== REGRESSION (HomeValue) ===')
print(drift_check('MedInc (no shift)',                 reg_train_feat, reg_train_feat))
print(drift_check('MedInc (10% upward shift)',         reg_train_feat, reg_prod_modest))
print(drift_check('MedInc (20% upward shift)',         reg_train_feat, reg_prod_bigger))

# --- Visualize the bigger-shift scenarios side by side ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(clf_train_feat,  bins=30, alpha=0.5, color=CLF_COLOR, label='Training data')
axes[0].hist(clf_prod_bigger, bins=30, alpha=0.5, color=RED,       label='"Production" — bigger sensor drift')
axes[0].set_title("MedScreen — 'worst perimeter' (training vs bigger-shift production)", fontsize=12, fontweight='bold')
axes[0].set_xlabel('worst perimeter'); axes[0].set_ylabel('count'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].hist(reg_train_feat,  bins=30, alpha=0.5, color=REG_COLOR, label='Training data')
axes[1].hist(reg_prod_bigger, bins=30, alpha=0.5, color=RED,       label='"Production" — 20% income shift')
axes[1].set_title("HomeValue — 'MedInc' (training vs 20%-shift production)", fontsize=12, fontweight='bold')
axes[1].set_xlabel('MedInc (10K USD units)'); axes[1].set_ylabel('count'); axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('Drift detection in code — the bigger shifts that crossed the 0.10 alert threshold', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Reading the output:**

Six rows total, three per case. Two clear patterns to read off.

**The "no shift" rows are the sanity check.** Both come back at `KS=0.000` with `p=1` and the `✓ OK` flag. That has to be true — comparing a distribution to itself can never look different, so any failure here would be a bug in the code, not a finding. Always include this row in monitoring scripts so a regression in the *test infrastructure itself* shows up immediately, before you stake any deployment decision on the test's other rows.

**MedScreen — small samples, the noise floor of drift detection.** The "small sensor drift" row (random noise with std 5, on top of `worst perimeter` whose own std is about 33) lands at **KS \~ 0.031** with `p ~ 0.98` — well below the 0.10 alert threshold, so `✓ OK`. The "bigger sensor drift" row (random noise with std 20) lands at **KS \~ 0.108** with `p ~ 0.010` — just above the threshold, so `🚨 DRIFT`. With only 455 training patients, KS values move noticeably between scenarios, and the bar to cross the threshold sits well within what a realistic microscope recalibration could produce. The lesson for the State Health Department's review board: small sensor changes blur into measurement noise; a microscope swap large enough to perceptibly shift the readings is exactly what the test should catch — but on a 455-patient pool you need a real shift to see it.

**HomeValue — what large sample sizes do to p-values.** The "10% upward shift" row lands at **KS \~ 0.090** with a p-value of **about 3 × 10⁻⁵⁹** — `✓ OK`. The "20% upward shift" row lands at **KS \~ 0.167** with a p-value near **8 × 10⁻²⁰³** — `🚨 DRIFT`. Both p-values are vanishingly small even though the first scenario falls *below* the alert threshold. The KS statistic itself measures the *size* of the shift; the p-value answers a different question — *"could this shift be pure sampling noise?"* — and on 16,512 tracts the answer is always *"no, with extreme confidence,"* regardless of whether the shift is operationally meaningful. **That is why the alert threshold is set on the KS statistic, not on the p-value.** A 10% income shift across the whole state is real (it shows up unambiguously in the data) but it is not yet big enough to demand an immediate model re-train; a 20% shift is. The business judgment lives in choosing the threshold on the magnitude metric, not on the certainty metric.

> **A question that often comes up here:** *"if the p-value is essentially zero, shouldn't that always trigger an alert?"* No — and that intuition is the most common mistake when teams first set up drift monitoring. p-values are about *evidence that the distributions differ*, not about *how big the difference is*. With enough samples, any non-identical pair of distributions will produce a tiny p-value — even a shift too small to care about operationally. **Always alert on the KS statistic's magnitude** (or the equivalent magnitude statistic for whichever drift test you choose); use the p-value only as a sanity check that the test ran cleanly.

> **A question that often comes up here:** *"how do the two business cases compare?"* The MedScreen rows show how hard it is to detect small drifts on small datasets — you need a bigger physical change before the test sees it. The HomeValue rows show the opposite — on a 16,500-tract training set the test sees every shift with overwhelming statistical confidence, so the threshold has to do the work of separating *real* from *operationally meaningful*. Both lessons matter when you set the thresholds on your own monitoring card.

---

### 7.5 Putting It All on One Page — The Monitoring Card

Each model you deploy gets exactly **one page** of monitoring documentation. Four sections:

1. **Which model is running, and why?** Model class, hyperparameters, the champion statement, and the §5 CV record — the five fold scores, their mean and their SD. *(For MedScreen: `LogReg(C=1.0)` inside a `StandardScaler` pipeline, CV ROC-AUC mean 0.9941, SD 0.0072, cross-validated Brier 0.0247. For HomeValue: `RandomForestRegressor()` with scikit-learn's defaults, CV R² mean 0.8036, SD 0.0117, CV RMSE about USD 51,000 — and a note that the choice over the tuned GBM is margin-sensitive.)*
2. **What did the test set say, and when?** The test-set number, the deployment bar it was judged against, the INSIDE / ABOVE / BELOW description from §6, and the date the envelope was opened. One line that proves the model was frozen before anyone looked at the test set.
3. **What are we tracking, and what is the alert threshold?** The condensed monitoring checklist from §7.2 (for MedScreen) or §7.3 (for HomeValue) — just the actionable rows, each with the **business alert threshold** declared at deployment.
4. **Who acts, and how?** Who runs the monthly checks, who co-signs the quarterly review, and who has the authority to escalate to Step 2 or Step 3.

The same one-page template works for every model your team ships across your career. Draft it once at deployment time and it is ready every quarter when a stakeholder asks *"how do we know the model is still working?"* It is also the document you hand to any new analyst inheriting the model — one page they can read in five minutes, instead of a meeting they need to schedule.

> **A question that often comes up here:** *"how much drift is 'enough' to act?"* Declare it in business terms before you ship, and write it on the card: for HomeValue, the typical error the deployment council would no longer accept (the USD 70,000 RMSE in §7.3); for MedScreen, the recall the review board would no longer accept (0.92 in §7.2). **Act when a production metric crosses its declared threshold for two consecutive monitoring periods** — one bad month by itself can be ordinary noise, two in a row is a signal, and that is when you start Step 1 (re-evaluate). Why not use the worst of the five CV fold scores as the trigger? Because fold scores describe how the development evidence scattered, not what the business can tolerate, and the worst of five numbers is itself a noisy statistic. The drift-test thresholds in the monitoring checklists (0.10 / 0.15 in the KS tests above) are conservative starting values that work fine before you have any production history; once you have six months of monitoring data, you can tune them up or down based on what actually counts as "unusual" in your specific business context.

---

## 8. Wrap-Up — Key Takeaways

**What landed today:**

1. **The selection protocol is declared before any score exists.** The criteria block fixes the roster (every reference model from nb01–nb13, with the configuration its notebook carried forward), the incumbent, the ladder order, the margin band, and the deployment bar.
2. **Fold scores describe; the ladder decides.** Each candidate is reported by its five fold scores, mean, and SD — no per-model interval. Starting from the model the stakeholder runs today, each challenger faces the champion of the moment with `compare_paired`, and only *"A is better by a worthwhile amount"* promotes it.
3. **The champion is stated in four points** — champion, evidence, checks, and what would change our mind — and the band (small / base / large) and the closest rival's switching cutoff show how much the choice depends on the margin.
4. **The locked test set opens exactly once per case, for the frozen champion**, refit on all of `X_train`. The deployment bar declared in advance makes the ship decision; INSIDE / ABOVE / BELOW only describes the test score against the CV mean ± δ. Two ceremonies in this notebook is a teaching demo; in any single real business analysis, the test set opens once and only once.
5. **Monitoring is the second half of the job.** §7 builds the one-page monitoring card every deployed model needs: *what to track*, *how often*, *the business alert threshold*, and *which step (Re-evaluate, Re-train, or Rebuild) you escalate to when an alert fires*.

**The two champions, named:**

- **MedScreen — `LogisticRegression(C=1.0)`, the incumbent, kept.** Every challenger's comparison with it is unresolved, robustly across the band; with two folds at 0.9995 this evidence could not certify a worthwhile gain for any challenger, even a perfect one. The explainable model stays because the evidence cannot show anything better — not because it has been proven the best. Test ROC-AUC 0.9993, above the 0.95 bar.
- **HomeValue — the random forest (scikit-learn's defaults), promoted.** It beat OLS by a worthwhile amount (+0.1992 $R^2$, [+0.1800, +0.2183]); the tuned GBM — the highest CV mean on the roster — is unresolved against it at the CFO's base margin ([+0.0091, +0.0271]), so the forest keeps the job. The call is **margin-sensitive**: a CFO who would switch for 0.005 $R^2$ would get the GBM, and the statement says so. Test $R^2$ 0.8005, above the 0.75 bar. The tuned GBM is the first candidate for the next evaluation, not a fallback.

**What comes next — nb16 now, the packaging work in the poster studios:**

The champions are now committed. Next up is **nb16 (Time-Series Forecasting)**, which carries this same CV-first discipline to data whose folds have to respect time — the forecasting tools your group will use in the **Course Case Competition**, a group S&P 500 forecasting competition on Kaggle that starts with nb18. The packaging work runs in parallel on your own dataset: the Friday poster studios and milestones **M08 (Complex Models)**, **M09 (Poster First Draft)**, and **M10 (Final Poster)** turn the protocol you just ran into a complete deployment package — declaring your stakeholder's margins before scoring, reporting the incumbent baseline's five fold scores with their mean and SD, tuning a more complex model with `GridSearchCV` / `RandomizedSearchCV` (the best cell is a nominee), climbing the ladder with `compare_paired`, writing the champion statement, and retraining the champion on the full training split with `random_state` locked so the fitted Pipeline reproduces cleanly every time you re-execute the modeling cell. nb17 shows how to put the paired-gain chart and the champion statement on the poster.

> **A question that often comes up at this point:** *"if nb14 already named the champions, why is there still modeling work left in the milestones?"* Because a complete deployment package is more than naming a model, and your poster milestones ask for the whole package on **your** dataset: (a) a criteria block written before any score, (b) the incumbent baseline's fold scores, (c) a systematic hyperparameter search whose winner is treated as a nominee, (d) a ladder of `compare_paired` comparisons against your declared margin, (e) a four-point champion statement, (f) a retrain on the full training split with `random_state` locked, and (g) the required visualizations and a polished 250-word abstract. nb14 ran that protocol on two demo cases with a roster inherited from earlier notebooks; M08 through M10 and the Friday poster studios turn it into a complete walkthrough on the data your group actually chose.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (check MedScreen's champion + write its statement) and Exercise 2 (check HomeValue's champion + write its statement).
2. **Run All Cells** — `Runtime → Run all` to execute every cell including the two §6 ceremonies.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 14 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both exercise code cells print the band, the closest rival with its switching cutoff, and the checks
- [ ] Both champion statements are written in the stakeholder's language (four points each)
- [ ] Both ceremonies print the deployment-bar line and render the money plot
- [ ] You can defend the singleness rule in plain English

### Next Step:

- **Notebook 16** — Time-Series Forecasting: the same CV-first discipline applied to data whose folds have to respect time. The packaging work on your own dataset continues in parallel in the Friday poster studios and milestones **M08**, **M09**, and **M10**.

---

---

## Bibliography

- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning with Python* — Ch. 5 (Resampling Methods). Springer.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* — Chapters on model assessment and selection.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* — Chapter on evaluation and decision protocols.
- Cawley, G. C., & Talbot, N. L. C. (2010). "On over-fitting in model selection and subsequent selection bias in performance evaluation." *Journal of Machine Learning Research*, 11, 2079–2107.
- Dietterich, T. G. (1998). "Approximate statistical tests for comparing supervised classification learning algorithms." *Neural Computation*, 10(7), 1895–1923.
- Nadeau, C., & Bengio, Y. (2003). "Inference for the generalization error." *Machine Learning*, 52, 239–281 — the correction for fold overlap in `compare_paired`.
- Lakens, D. (2017). "Equivalence tests: A practical primer for t tests, correlations, and meta-analyses." *Social Psychological and Personality Science*, 8(4), 355–362.
- Smith, J. E., & Winkler, R. L. (2006). "The optimizer's curse: Skepticism and postdecision surprise in decision analysis." *Management Science*, 52(3), 311–322.
- Thomas, P. S., Theocharous, G., & Ghavamzadeh, M. (2015). "High confidence policy improvement." *Proceedings of the 32nd International Conference on Machine Learning*, PMLR 37, 2380–2388 — promote a new policy only when the lower limit of its improvement clears a threshold.
- Mitchell, M., Wu, S., Zaldivar, A., Barnes, P., Vasserman, L., Hutchinson, B., Spitzer, E., Raji, I. D., & Gebru, T. (2019). "Model Cards for Model Reporting." *Proceedings of the Conference on Fairness, Accountability, and Transparency (FAT\* '19)*, 220–229.
- Sculley, D., Holt, G., Golovin, D., Davydov, E., Phillips, T., Ebner, D., Chaudhary, V., Young, M., Crespo, J.-F., & Dennison, D. (2015). "Hidden Technical Debt in Machine Learning Systems." *Advances in Neural Information Processing Systems (NeurIPS)*, 28.
- Provost, F., & Fawcett, T. (2013). *Data Science for Business* — Chapter on deployment, monitoring, and the model lifecycle.
- scikit-learn User Guide — [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)
- scikit-learn User Guide — [Common pitfalls in interpretation and evaluation](https://scikit-learn.org/stable/common_pitfalls.html)

---

<center>

**Thank you!**

</center>